# Bibliotecas

In [1]:
# Bibliotecas gerais
import os
import time
from datetime import datetime
from getpass import getpass
from IPython.display import display
import html
from pathlib import Path
import json
import math
import shutil
import unicodedata
import re
import warnings
warnings.filterwarnings("ignore")

# Bibliotecas de dados
import pandas as pd
pd.set_option('display.precision', 10)
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
from scipy.stats import mannwhitneyu, wilcoxon

# Biblioteca para armazenar dados no banco de dados
from sqlalchemy import create_engine, text

# Configurações banco de dados

In [2]:
# Tenta recuperar a conexão com o banco, se definido antes
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "MF"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    print("Configuração do Banco de Dados criada com Sucesso!!!")

else:
    print("Configuração já existe. Usando configuração já existente!!!")

Digite seu usuário MySQL:  tomida
Digite sua senha MySQL (não será exibida):  ········


Configuração do Banco de Dados criada com Sucesso!!!


# 1) Carga e Preparação dos Dados

In [3]:
# ==========================================================
# ETAPA 1 - CARGA E PREPARAÇÃO DOS DADOS
# ==========================================================
"""
Descrição:
     Esta etapa:
     1) Lê os dados históricos de preços do SQL
     2) Lê os dados cadastrais das empresas
     3) Padroniza datas e preços
     4) Monta a base mensal de preços
     5) Calcula retornos mensais dos ativos
     6) Baixa e prepara o Ibovespa mensal via yfinance
     7) Cria os diretórios de saída do projeto

 Pré-requisito:
     A variável db_engine já deve existir no notebook.

 Outputs principais:
     - df_cotacoes_raw
     - df_info_empresas
     - df_precos_mensais
     - df_retornos_mensais
     - df_ibov_mensal
     - serie_retorno_ibov
"""
     
# ==========================================================
# 1) CRIAÇÃO DAS PASTAS DO PROJETO
# ==========================================================
print("=" * 80)
print("CRIANDO ESTRUTURA DE PASTAS DO PROJETO")
print("=" * 80)

pastas_projeto = [
    "resultados",
    "resultados/graficos",
    "resultados/tabelas",
    "resultados/dados_intermediarios",
    "resultados/html"
]

for pasta in pastas_projeto:
    os.makedirs(pasta, exist_ok=True)

print("Pastas criadas/verificadas com sucesso.")

# ==========================================================
# 2) LEITURA DAS TABELAS DO BANCO
# ==========================================================
print("\n" + "=" * 80)
print("LENDO DADOS DO BANCO")
print("=" * 80)

query_cotacoes = """
SELECT
    ticker,
    data,
    open,
    high,
    low,
    close,
    close_adj,
    average,
    volume_qtd,
    volume_fin,
    negocios
FROM cot_hist_consolidado
"""

query_info = """
SELECT
    ticker,
    issuer_code,
    tipo_acao,
    isin,
    nome,
    nom_res,
    cnpj,
    setor,
    subsetor,
    segmento,
    nome_pregao,
    segmento_negociacao,
    num_acoes,
    ultimo_balanco_processado
FROM dados_info_empresa
"""

df_cotacoes_raw = pd.read_sql(query_cotacoes, db_engine)
df_info_empresas = pd.read_sql(query_info, db_engine)

print(f"df_cotacoes_raw: {df_cotacoes_raw.shape}")
print(f"df_info_empresas: {df_info_empresas.shape}")

# ==========================================================
# 3) PADRONIZAÇÃO DAS BASES
# ==========================================================
print("\n" + "=" * 80)
print("PADRONIZANDO BASES")
print("=" * 80)

# -----------------------------
# Cotacoes
# -----------------------------
df_cotacoes_raw["ticker"] = df_cotacoes_raw["ticker"].astype(str).str.strip().str.upper()
df_cotacoes_raw["data"] = pd.to_datetime(df_cotacoes_raw["data"], errors="coerce")

# Preço principal: close_adj; fallback para close
df_cotacoes_raw["preco"] = df_cotacoes_raw["close_adj"]
mask_preco_nulo = df_cotacoes_raw["preco"].isna()
df_cotacoes_raw.loc[mask_preco_nulo, "preco"] = df_cotacoes_raw.loc[mask_preco_nulo, "close"]

# Remove linhas sem ticker, data ou preço
df_cotacoes_raw = df_cotacoes_raw.dropna(subset=["ticker", "data", "preco"]).copy()

# Remove duplicatas exatas
df_cotacoes_raw = df_cotacoes_raw.drop_duplicates()

# -----------------------------
# Info empresas
# -----------------------------
df_info_empresas["ticker"] = df_info_empresas["ticker"].astype(str).str.strip().str.upper()

# Remove duplicatas por ticker, mantendo o último registro disponível
df_info_empresas = (
    df_info_empresas
    .drop_duplicates(subset=["ticker"], keep="last")
    .copy()
)

# Padroniza classificação setorial
for col in ["setor", "subsetor", "segmento"]:
    if col in df_info_empresas.columns:
        df_info_empresas[col] = (
            df_info_empresas[col]
            .astype(str)
            .str.strip()
            .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        )

print("Padronização concluída.")

# ==========================================================
# 4) MONTAGEM DA BASE MENSAL DE PREÇOS
# ==========================================================
print("\n" + "=" * 80)
print("MONTANDO BASE MENSAL DE PREÇOS")
print("=" * 80)

df_precos = df_cotacoes_raw.copy()

# Cria coluna de referência mensal
df_precos["ano_mes"] = df_precos["data"].dt.to_period("M").dt.to_timestamp()

# Ordena para pegar o último preço disponível de cada mês por ticker
df_precos = df_precos.sort_values(["ticker", "data"])

df_precos_mensais = (
    df_precos
    .groupby(["ticker", "ano_mes"], as_index=False)
    .last()
    .rename(columns={"ano_mes": "data_ref"})
)

# Mantém apenas colunas relevantes
colunas_mensais = [
    "ticker", "data_ref", "data", "preco",
    "close", "close_adj", "volume_qtd", "volume_fin", "negocios"
]
colunas_mensais = [c for c in colunas_mensais if c in df_precos_mensais.columns]

df_precos_mensais = df_precos_mensais[colunas_mensais].copy()

print(f"df_precos_mensais: {df_precos_mensais.shape}")
print("\nAmostra da base mensal:")
display(df_precos_mensais.head())

# ==========================================================
# 5) CÁLCULO DOS RETORNOS MENSAIS DOS ATIVOS
# ==========================================================
print("\n" + "=" * 80)
print("CALCULANDO RETORNOS MENSAIS DOS ATIVOS")
print("=" * 80)

df_retornos_mensais = df_precos_mensais[["ticker", "data_ref", "preco"]].copy()
df_retornos_mensais = df_retornos_mensais.sort_values(["ticker", "data_ref"])

df_retornos_mensais["retorno_mensal"] = (
    df_retornos_mensais
    .groupby("ticker")["preco"]
    .pct_change()
)

print(f"df_retornos_mensais: {df_retornos_mensais.shape}")
print("\nAmostra dos retornos mensais:")
display(df_retornos_mensais.head(10))

# ==========================================================
# 6) ENRIQUECIMENTO COM CLASSIFICAÇÃO SETORIAL
# ==========================================================
print("\n" + "=" * 80)
print("ENRIQUECENDO BASES COM DADOS CADASTRAIS")
print("=" * 80)

df_precos_mensais = df_precos_mensais.merge(
    df_info_empresas[["ticker", "issuer_code", "setor", "subsetor", "segmento", "nome_pregao"]],
    on="ticker",
    how="left"
)

df_retornos_mensais = df_retornos_mensais.merge(
    df_info_empresas[["ticker", "issuer_code", "setor", "subsetor", "segmento", "nome_pregao"]],
    on="ticker",
    how="left"
)

print("Merge com dados cadastrais concluído.")
print("\nPercentual de tickers com setor preenchido:")
pct_setor = df_info_empresas["setor"].notna().mean() * 100
print(f"{pct_setor:.2f}%")

# ==========================================================
# 7) DOWNLOAD E PREPARAÇÃO DO IBOVESPA - VERSÃO CORRIGIDA
# ==========================================================
print("\n" + "=" * 80)
print("BAIXANDO E PREPARANDO O IBOVESPA (^BVSP)")
print("=" * 80)

data_inicial_ibov = "2010-01-01"

ibov = yf.download("^BVSP", start=data_inicial_ibov, progress=False, auto_adjust=False)

if ibov.empty:
    raise ValueError("Não foi possível baixar os dados do Ibovespa via yfinance.")

ibov = ibov.reset_index()

# ----------------------------------------------------------
# Trata colunas MultiIndex do yfinance
# ----------------------------------------------------------
if isinstance(ibov.columns, pd.MultiIndex):
    novas_colunas = []
    for col in ibov.columns:
        parte_0 = str(col[0]).strip() if col[0] is not None else ""
        parte_1 = str(col[1]).strip() if col[1] is not None else ""

        # Se a segunda parte estiver vazia, usa só a primeira
        if parte_1 in ["", "None", "nan"]:
            novas_colunas.append(parte_0)
        else:
            novas_colunas.append(parte_0)

    ibov.columns = novas_colunas
else:
    ibov.columns = [str(c).strip() for c in ibov.columns]

print("Colunas do IBOV após padronização:")
print(list(ibov.columns))

# ----------------------------------------------------------
# Identifica colunas principais
# ----------------------------------------------------------
col_data = None
for c in ibov.columns:
    if c.lower() == "date":
        col_data = c
        break

if col_data is None:
    raise ValueError("Não foi possível identificar a coluna de data do Ibovespa.")

col_preco_ibov = None
for candidata in ["Adj Close", "Close"]:
    if candidata in ibov.columns:
        col_preco_ibov = candidata
        break

if col_preco_ibov is None:
    raise ValueError(
        f"Não foi possível identificar a coluna de preço do Ibovespa. Colunas disponíveis: {list(ibov.columns)}"
    )

df_ibov = ibov[[col_data, col_preco_ibov]].copy()
df_ibov.columns = ["data", "preco_ibov"]

df_ibov["data"] = pd.to_datetime(df_ibov["data"], errors="coerce")
df_ibov["preco_ibov"] = pd.to_numeric(df_ibov["preco_ibov"], errors="coerce")

df_ibov = df_ibov.dropna(subset=["data", "preco_ibov"]).copy()
df_ibov = df_ibov.sort_values("data")

df_ibov["ano_mes"] = df_ibov["data"].dt.to_period("M").dt.to_timestamp()

df_ibov_mensal = (
    df_ibov
    .groupby("ano_mes", as_index=False)
    .last()
    .rename(columns={"ano_mes": "data_ref"})
)

df_ibov_mensal["retorno_mensal_ibov"] = df_ibov_mensal["preco_ibov"].pct_change()

serie_retorno_ibov = (
    df_ibov_mensal
    .set_index("data_ref")["retorno_mensal_ibov"]
    .copy()
)

print(f"df_ibov_mensal: {df_ibov_mensal.shape}")
display(df_ibov_mensal.head())

# ==========================================================
# 8) CHECAGENS RÁPIDAS DE CONSISTÊNCIA
# ==========================================================
print("\n" + "=" * 80)
print("CHECAGENS RÁPIDAS")
print("=" * 80)

n_tickers_total = df_retornos_mensais["ticker"].nunique()
n_tickers_com_retorno = df_retornos_mensais.loc[
    df_retornos_mensais["retorno_mensal"].notna(), "ticker"
].nunique()

data_min = df_retornos_mensais["data_ref"].min()
data_max = df_retornos_mensais["data_ref"].max()

print(f"Total de tickers na base mensal: {n_tickers_total}")
print(f"Tickers com ao menos 1 retorno mensal válido: {n_tickers_com_retorno}")
print(f"Período da base de ativos: {data_min.date()} até {data_max.date()}")
print(f"Período do Ibovespa mensal: {df_ibov_mensal['data_ref'].min().date()} até {df_ibov_mensal['data_ref'].max().date()}")

# ==========================================================
# 9) SALVAMENTO DOS DADOS INTERMEDIÁRIOS
# ==========================================================
print("\n" + "=" * 80)
print("SALVANDO DADOS INTERMEDIÁRIOS")
print("=" * 80)

df_precos_mensais.to_parquet("resultados/dados_intermediarios/df_precos_mensais.parquet", index=False)
df_retornos_mensais.to_parquet("resultados/dados_intermediarios/df_retornos_mensais.parquet", index=False)
df_info_empresas.to_parquet("resultados/dados_intermediarios/df_info_empresas.parquet", index=False)
df_ibov_mensal.to_parquet("resultados/dados_intermediarios/df_ibov_mensal.parquet", index=False)

print("Arquivos salvos com sucesso em resultados/dados_intermediarios/")

# ==========================================================
# 10) RESUMO FINAL DA ETAPA
# ==========================================================
print("\n" + "=" * 80)
print("ETAPA 1 CONCLUÍDA")
print("=" * 80)

print("Objetos gerados:")
print("- df_cotacoes_raw")
print("- df_info_empresas")
print("- df_precos_mensais")
print("- df_retornos_mensais")
print("- df_ibov_mensal")
print("- serie_retorno_ibov")

CRIANDO ESTRUTURA DE PASTAS DO PROJETO
Pastas criadas/verificadas com sucesso.

LENDO DADOS DO BANCO
df_cotacoes_raw: (1262516, 11)
df_info_empresas: (698, 14)

PADRONIZANDO BASES
Padronização concluída.

MONTANDO BASE MENSAL DE PREÇOS
df_precos_mensais: (76331, 9)

Amostra da base mensal:


,ticker,data_ref,data,preco,close,close_adj,volume_qtd,volume_fin,negocios
0,AALR3,2016-10-01,2016-10-31,17.7157,18.06,17.7157,2523300,45857231.0,4238
1,AALR3,2016-11-01,2016-11-30,15.5457,15.85,15.5457,766100,12033095.0,877
2,AALR3,2016-12-01,2016-12-29,14.3718,14.65,14.3718,372300,5375545.0,1339
3,AALR3,2017-01-01,2017-01-31,13.6715,13.94,13.6715,223400,3116406.0,494
4,AALR3,2017-02-01,2017-02-24,12.4977,12.74,12.4977,238700,3077805.0,643



CALCULANDO RETORNOS MENSAIS DOS ATIVOS
df_retornos_mensais: (76331, 4)

Amostra dos retornos mensais:


,ticker,data_ref,preco,retorno_mensal
0,AALR3,2016-10-01,17.7157,NaN
1,AALR3,2016-11-01,15.5457,-0.1224902205
2,AALR3,2016-12-01,14.3718,-0.0755128428
3,AALR3,2017-01-01,13.6715,-0.0487273689
4,AALR3,2017-02-01,12.4977,-0.0858574407
5,AALR3,2017-03-01,15.3286,0.2265136785
6,AALR3,2017-04-01,18.2286,0.1891888366
7,AALR3,2017-05-01,17.3902,-0.0459936583
8,AALR3,2017-06-01,17.2027,-0.0107819347
9,AALR3,2017-07-01,16.7194,-0.0280944270



ENRIQUECENDO BASES COM DADOS CADASTRAIS
Merge com dados cadastrais concluído.

Percentual de tickers com setor preenchido:
91.69%

BAIXANDO E PREPARANDO O IBOVESPA (^BVSP)
Colunas do IBOV após padronização:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
df_ibov_mensal: (196, 4)


,data_ref,data,preco_ibov,retorno_mensal_ibov
0,2010-01-01,2010-01-29,65402.0,NaN
1,2010-02-01,2010-02-26,66503.0,0.0168343476
2,2010-03-01,2010-03-31,70372.0,0.0581778266
3,2010-04-01,2010-04-30,67530.0,-0.0403853805
4,2010-05-01,2010-05-31,63047.0,-0.0663853102



CHECAGENS RÁPIDAS
Total de tickers na base mensal: 698
Tickers com ao menos 1 retorno mensal válido: 687
Período da base de ativos: 2010-01-01 até 2025-12-01
Período do Ibovespa mensal: 2010-01-01 até 2026-04-01

SALVANDO DADOS INTERMEDIÁRIOS
Arquivos salvos com sucesso em resultados/dados_intermediarios/

ETAPA 1 CONCLUÍDA
Objetos gerados:
- df_cotacoes_raw
- df_info_empresas
- df_precos_mensais
- df_retornos_mensais
- df_ibov_mensal
- serie_retorno_ibov


# 2) Definição do Universo Elegível

In [4]:
# ==========================================================
# ETAPA 2 - UNIVERSO ELEGÍVEL
# ==========================================================
"""
Descrição:
    Esta etapa define quais ativos poderão participar das carteiras.

Parâmetros:
    - DATA_INICIAL_ANALISE: início da análise
    - MIN_MESES_VALIDOS: mínimo de meses com dados
    - MIN_PRESENCA: percentual mínimo de presença
    - COL_CLASSIFICACAO_SETORIAL: coluna usada para setor

Processo:
    - Filtra dados a partir de 2010
    - Remove preços inválidos
    - Calcula quantidade de meses por ativo
    - Calcula presença relativa
    - Filtra ativos com setor válido

Retorno:
    - df_elegibilidade_ativos
    - lista_ativos_elegiveis
    - df_retornos_elegiveis
    - df_precos_elegiveis
"""

print("="*80)
print("ETAPA 2 - UNIVERSO ELEGÍVEL")
print("="*80)

# =========================
# CONFIGURAÇÕES
# =========================
DATA_INICIAL_ANALISE = pd.Timestamp("2010-01-01")
MIN_MESES_VALIDOS = 60
MIN_PRESENCA = 0.80
COL_CLASSIFICACAO_SETORIAL = "setor"

# =========================
# BASE
# =========================
df_base = df_retornos_mensais.copy()
df_base = df_base[df_base["data_ref"] >= DATA_INICIAL_ANALISE]
df_base = df_base[df_base["preco"].notna()]
df_base = df_base[df_base["preco"] > 0]

# =========================
# ESTATÍSTICAS POR ATIVO
# =========================
stats = df_base.groupby("ticker").agg(
    data_inicio=("data_ref", "min"),
    data_fim=("data_ref", "max"),
    n_meses_validos=("data_ref", "nunique")
).reset_index()

stats["n_meses_total"] = (
    (stats["data_fim"].dt.to_period("M") - stats["data_inicio"].dt.to_period("M")).apply(lambda x: x.n) + 1
)

stats["presenca"] = stats["n_meses_validos"] / stats["n_meses_total"]

# =========================
# JUNTA SETOR
# =========================
stats = stats.merge(
    df_info_empresas[["ticker", COL_CLASSIFICACAO_SETORIAL]],
    on="ticker",
    how="left"
)

# =========================
# FILTROS
# =========================
stats["flag_mes"] = stats["n_meses_validos"] >= MIN_MESES_VALIDOS
stats["flag_presenca"] = stats["presenca"] >= MIN_PRESENCA
stats["flag_setor"] = stats[COL_CLASSIFICACAO_SETORIAL].notna()

stats["elegivel"] = (
    stats["flag_mes"] &
    stats["flag_presenca"] &
    stats["flag_setor"]
)

df_elegibilidade_ativos = stats.copy()

lista_ativos_elegiveis = (
    df_elegibilidade_ativos
    .loc[df_elegibilidade_ativos["elegivel"], "ticker"]
    .sort_values()
    .unique()
    .tolist()
)

# =========================
# BASE FINAL FILTRADA
# =========================
df_retornos_elegiveis = df_retornos_mensais[
    df_retornos_mensais["ticker"].isin(lista_ativos_elegiveis)
].copy()

df_precos_elegiveis = df_precos_mensais[
    df_precos_mensais["ticker"].isin(lista_ativos_elegiveis)
].copy()

# =========================
# LOGS
# =========================
print(f"Total de ativos na base: {df_base['ticker'].nunique()}")
print(f"Ativos elegíveis: {len(lista_ativos_elegiveis)}")

print("\nResumo filtros:")
print(df_elegibilidade_ativos[["flag_mes","flag_presenca","flag_setor","elegivel"]].mean())

display(df_elegibilidade_ativos.head())

ETAPA 2 - UNIVERSO ELEGÍVEL
Total de ativos na base: 698
Ativos elegíveis: 398

Resumo filtros:
flag_mes         0.6934097421
flag_presenca    0.8151862464
flag_setor       0.9169054441
elegivel         0.5702005731
dtype: float64


,ticker,data_inicio,data_fim,n_meses_validos,n_meses_total,presenca,setor,flag_mes,flag_presenca,flag_setor,elegivel
0,AALR3,2016-10-01,2025-12-01,111,111,1.0,Saúde,True,True,True,True
1,ABCB4,2010-01-01,2025-12-01,192,192,1.0,Financeiro,True,True,True,True
2,ABEV3,2013-11-01,2025-12-01,146,146,1.0,Consumo não Cíclico,True,True,True,True
3,ABYA3,2010-01-01,2010-02-01,2,2,1.0,Consumo Cíclico,False,True,True,False
4,ADHM3,2017-01-01,2020-12-01,48,48,1.0,NaN,False,True,False,False


# 3) Carteiras Aleatórias

In [5]:
# ==========================================================
# ETAPA 3 - CARTEIRAS ALEATÓRIAS
# ==========================================================
"""
Descrição:
    Gera carteiras aleatórias com pesos iguais.

Parâmetros:
    - tamanhos_carteira
    - n_carteiras_por_tamanho

Processo:
    - Sorteia ativos sem reposição
    - Define pesos iguais
    - Armazena composição das carteiras

Retorno:
    - df_carteiras_aleatorias
    - dicionario_carteiras_aleatorias
"""

print("="*80)
print("ETAPA 3 - CARTEIRAS ALEATÓRIAS")
print("="*80)

# =========================
# CONFIGURAÇÕES
# =========================
tamanhos_carteira = [5, 10, 15, 20, 25, 30]
n_carteiras_por_tamanho = 200

ativos = np.array(lista_ativos_elegiveis)

# opcional: fixa seed pra reprodutibilidade
np.random.seed(42)

# =========================
# GERAÇÃO
# =========================
registros = []
dicionario_carteiras_aleatorias = {}

id_global = 0

for n_ativos in tamanhos_carteira:

    print(f"Gerando carteiras com {n_ativos} ativos...")

    for i in range(n_carteiras_por_tamanho):

        id_global += 1

        ativos_sorteados = np.random.choice(
            ativos,
            size=n_ativos,
            replace=False
        )

        peso = 1.0 / n_ativos

        nome = f"RANDOM_N{n_ativos}_{i:03d}"

        dicionario_carteiras_aleatorias[nome] = {
            "ativos": ativos_sorteados,
            "pesos": np.repeat(peso, n_ativos),
            "n_ativos": n_ativos
        }

        for ativo in ativos_sorteados:
            registros.append({
                "carteira": nome,
                "ticker": ativo,
                "peso": peso,
                "n_ativos": n_ativos
            })

df_carteiras_aleatorias = pd.DataFrame(registros)

# =========================
# VALIDAÇÃO
# =========================
print("\nResumo:")
print(df_carteiras_aleatorias.groupby("n_ativos")["carteira"].nunique())

print("\nTotal carteiras:")
print(df_carteiras_aleatorias["carteira"].nunique())

display(df_carteiras_aleatorias.head())

# =========================
# SALVAR
# =========================
df_carteiras_aleatorias.to_parquet(
    "resultados/dados_intermediarios/df_carteiras_aleatorias.parquet",
    index=False
)

print("\nCarteiras geradas com sucesso.")

ETAPA 3 - CARTEIRAS ALEATÓRIAS
Gerando carteiras com 5 ativos...
Gerando carteiras com 10 ativos...
Gerando carteiras com 15 ativos...
Gerando carteiras com 20 ativos...
Gerando carteiras com 25 ativos...
Gerando carteiras com 30 ativos...

Resumo:
n_ativos
5     200
10    200
15    200
20    200
25    200
30    200
Name: carteira, dtype: int64

Total carteiras:
1200


,carteira,ticker,peso,n_ativos
0,RANDOM_N5_000,HYPE3,0.2,5
1,RANDOM_N5_000,YDUQ3,0.2,5
2,RANDOM_N5_000,BBSE3,0.2,5
3,RANDOM_N5_000,ISAE4,0.2,5
4,RANDOM_N5_000,CESP6,0.2,5



Carteiras geradas com sucesso.


# 4) Carteiras Setoriais

In [6]:
# ==========================================================
# ETAPA 4 - CARTEIRAS SETORIAIS
# ==========================================================
"""
Descrição:
    Constrói carteiras por setor com pesos iguais.

Parâmetros:
    - COL_SETOR
    - MIN_ATIVOS_SETOR

Processo:
    - Agrupa ativos por setor
    - Filtra setores com poucos ativos
    - Define pesos iguais

Retorno:
    - df_carteiras_setoriais
    - dicionario_carteiras_setoriais
"""

print("="*80)
print("ETAPA 4 - CARTEIRAS SETORIAIS")
print("="*80)

COL_SETOR = "setor"
MIN_ATIVOS_SETOR = 5

df_setor = df_info_empresas[
    df_info_empresas["ticker"].isin(lista_ativos_elegiveis)
][["ticker", COL_SETOR]].dropna()

grupo = (
    df_setor.groupby(COL_SETOR)["ticker"]
    .apply(list)
    .reset_index()
)

grupo["n_ativos"] = grupo["ticker"].apply(len)
grupo = grupo[grupo["n_ativos"] >= MIN_ATIVOS_SETOR]

registros = []
dicionario_carteiras_setoriais = {}

for _, row in grupo.iterrows():

    setor = row[COL_SETOR]
    ativos = np.array(row["ticker"])
    n = len(ativos)
    peso = 1/n

    nome = f"SETOR_{setor}".upper().replace(" ", "_")

    dicionario_carteiras_setoriais[nome] = {
        "ativos": ativos,
        "pesos": np.repeat(peso, n),
        "n_ativos": n
    }

    for ativo in ativos:
        registros.append({
            "carteira": nome,
            "ticker": ativo,
            "peso": peso,
            "setor": setor,
            "n_ativos": n
        })

df_carteiras_setoriais = pd.DataFrame(registros)

print("\nResumo:")
print(df_carteiras_setoriais.groupby("carteira")["ticker"].count())

df_carteiras_setoriais.to_parquet(
    "resultados/dados_intermediarios/df_carteiras_setoriais.parquet",
    index=False
)

print("\nCarteiras setoriais ok.")

ETAPA 4 - CARTEIRAS SETORIAIS

Resumo:
carteira
SETOR_BENS_INDUSTRIAIS                   52
SETOR_COMUNICAÇÕES                        7
SETOR_CONSUMO_CÍCLICO                    88
SETOR_CONSUMO_NÃO_CÍCLICO                21
SETOR_FINANCEIRO                         76
SETOR_MATERIAIS_BÁSICOS                  44
SETOR_PETRÓLEO,_GÁS_E_BIOCOMBUSTÍVEIS    12
SETOR_SAÚDE                              21
SETOR_TECNOLOGIA_DA_INFORMAÇÃO           10
SETOR_UTILIDADE_PÚBLICA                  65
Name: ticker, dtype: int64

Carteiras setoriais ok.


# 5) Carteiras por Subsetor

In [7]:
# ==========================================================
# ETAPA 5 - CARTEIRAS SUBSETORIAIS
# ==========================================================
"""
Descrição:
    Constrói carteiras por subsetor com pesos iguais.

Parâmetros:
    - COL_SUBSETOR
    - MIN_ATIVOS_SUBSETOR

Processo:
    - Agrupa ativos por subsetor
    - Filtra subsetores pequenos
    - Define pesos iguais

Retorno:
    - df_carteiras_subsetoriais
    - dicionario_carteiras_subsetoriais
"""

print("="*80)
print("ETAPA 5 - CARTEIRAS SUBSETORIAIS")
print("="*80)

COL_SUBSETOR = "subsetor"
MIN_ATIVOS_SUBSETOR = 5

df_subsetor = df_info_empresas[
    df_info_empresas["ticker"].isin(lista_ativos_elegiveis)
][["ticker", COL_SUBSETOR]].dropna()

grupo = (
    df_subsetor.groupby(COL_SUBSETOR)["ticker"]
    .apply(list)
    .reset_index()
)

grupo["n_ativos"] = grupo["ticker"].apply(len)
grupo = grupo[grupo["n_ativos"] >= MIN_ATIVOS_SUBSETOR]

registros = []
dicionario_carteiras_subsetoriais = {}

for _, row in grupo.iterrows():

    subsetor = row[COL_SUBSETOR]
    ativos = np.array(row["ticker"])
    n = len(ativos)
    peso = 1/n

    nome = f"SUBSETOR_{subsetor}".upper().replace(" ", "_")

    dicionario_carteiras_subsetoriais[nome] = {
        "ativos": ativos,
        "pesos": np.repeat(peso, n),
        "n_ativos": n
    }

    for ativo in ativos:
        registros.append({
            "carteira": nome,
            "ticker": ativo,
            "peso": peso,
            "subsetor": subsetor,
            "n_ativos": n
        })

df_carteiras_subsetoriais = pd.DataFrame(registros)

print("\nResumo:")
print(df_carteiras_subsetoriais.groupby("carteira")["ticker"].count())

df_carteiras_subsetoriais.to_parquet(
    "resultados/dados_intermediarios/df_carteiras_subsetoriais.parquet",
    index=False
)

print("\nCarteiras subsetoriais ok.")

ETAPA 5 - CARTEIRAS SUBSETORIAIS

Resumo:
carteira
SUBSETOR_ALIMENTOS_PROCESSADOS                       12
SUBSETOR_COMÉRCIO_E_DISTRIBUIÇÃO                     11
SUBSETOR_COMÉRCIO_VAREJISTA                          20
SUBSETOR_CONSTRUÇÃO_CIVIL                            24
SUBSETOR_CONSTRUÇÃO_E_ENGENHARIA                      8
SUBSETOR_DIVERSOS                                     9
SUBSETOR_ENERGIA_ELÉTRICA                            57
SUBSETOR_EXPLORAÇÃO_DE_IMÓVEIS                       19
SUBSETOR_HOLDINGS_DIVERSIFICADAS                      7
SUBSETOR_INTERMEDIÁRIOS_FINANCEIROS                  39
SUBSETOR_MADEIRA_E_PAPEL                              7
SUBSETOR_MATERIAL_DE_TRANSPORTE                      10
SUBSETOR_MINERAÇÃO                                    6
SUBSETOR_MÁQUINAS_E_EQUIPAMENTOS                     13
SUBSETOR_PETRÓLEO,_GÁS_E_BIOCOMBUSTÍVEIS             12
SUBSETOR_PREVIDÊNCIA_E_SEGUROS                        8
SUBSETOR_PROGRAMAS_E_SERVIÇOS                        

# 6) Carteiras Markowitz

In [8]:
%%time
# ==========================================================
# ETAPA 6 - MARKOWITZ
# ==========================================================
"""
Descrição:
    Calcula pesos de Markowitz mensalmente, sem usar informação futura
    e sem perder meses por tratamento excessivamente restritivo de NaN.

Parâmetros:
    - JANELA_ESTIMACAO
    - LIMITE_PESO_MAX

Processo:
    - Para cada mês t:
        - usa os 60 meses anteriores
        - estima média e covariância
        - otimiza Sharpe
        - aplica no mês t
    - Preenche faltantes com 0 para manter a série contínua

Retorno:
    - df_ret_markowitz
"""

print("="*80)
print("ETAPA 6 - MARKOWITZ")
print("="*80)

# =========================
# CONFIGURAÇÕES
# =========================
JANELA_ESTIMACAO = 60
LIMITE_PESO_MAX = 0.30

# =========================
# BASE
# =========================
df_pivot = df_retornos_elegiveis.pivot(
    index="data_ref",
    columns="ticker",
    values="retorno_mensal"
).sort_index()

datas = df_pivot.index

# =========================
# FUNÇÃO OBJETIVO
# =========================
def neg_sharpe(w, mu, cov):
    vol = np.sqrt(w @ cov @ w)
    if vol == 0:
        return 1e6
    return -(w @ mu) / vol

# =========================
# LOOP PRINCIPAL
# =========================
registros = []

for i in range(JANELA_ESTIMACAO, len(datas)):

    data_atual = datas[i]
    df_hist = df_pivot.iloc[i-JANELA_ESTIMACAO:i]

    if i % 12 == 0:
        print(f"Processando {data_atual.date()}")

    for nome, info in dicionario_carteiras_aleatorias.items():

        ativos = info["ativos"]

        # histórico apenas passado
        df_sub = df_hist[ativos].copy()

        # mantém consistência com as aleatórias
        df_sub = df_sub.fillna(0)

        # garante que existe janela mínima
        if df_sub.shape[0] < 36:
            continue

        mu = df_sub.mean().values
        cov = df_sub.cov().values

        n = len(ativos)
        x0 = np.repeat(1/n, n)

        cons = {"type": "eq", "fun": lambda x: x.sum() - 1}
        bounds = [(0, LIMITE_PESO_MAX)] * n

        try:
            res = minimize(
                neg_sharpe,
                x0,
                args=(mu, cov),
                method="SLSQP",
                bounds=bounds,
                constraints=cons
            )

            if not res.success:
                continue

            pesos = res.x

            # retorno no mês atual
            ret_mes = df_pivot.loc[data_atual, ativos].copy()

            # mesma lógica das aleatórias: faltante = 0
            ret_mes = ret_mes.fillna(0)

            retorno = np.dot(pesos, ret_mes.values)

            registros.append({
                "data_ref": data_atual,
                "carteira": nome,
                "carteira_exibicao": f"MARKOWITZ_{nome}",
                "retorno": retorno,
                "tipo": "markowitz"
            })

        except Exception:
            continue

# =========================
# DATAFRAME FINAL
# =========================
df_ret_markowitz = pd.DataFrame(registros)

df_ret_markowitz = df_ret_markowitz.sort_values(["carteira", "data_ref"])

df_ret_markowitz["retorno_acum"] = (
    df_ret_markowitz
    .groupby("carteira")["retorno"]
    .transform(lambda x: (1 + x).cumprod())
)

# =========================
# VALIDAÇÃO
# =========================
print("\nCarteiras válidas:")
print(df_ret_markowitz["carteira"].nunique())

print("\nPeríodo:")
print(df_ret_markowitz["data_ref"].min(), "até", df_ret_markowitz["data_ref"].max())

display(df_ret_markowitz.head())

# =========================
# SALVAR
# =========================
df_ret_markowitz.to_parquet(
    "resultados/dados_intermediarios/df_ret_markowitz.parquet",
    index=False
)

print("\nMarkowitz concluído com sucesso.")

ETAPA 6 - MARKOWITZ
Processando 2015-01-01
Processando 2016-01-01
Processando 2017-01-01
Processando 2018-01-01
Processando 2019-01-01
Processando 2020-01-01
Processando 2021-01-01
Processando 2022-01-01
Processando 2023-01-01
Processando 2024-01-01
Processando 2025-01-01

Carteiras válidas:
1200

Período:
2015-01-01 00:00:00 até 2025-12-01 00:00:00


,data_ref,carteira,carteira_exibicao,retorno,tipo,retorno_acum
200,2015-01-01,RANDOM_N10_000,MARKOWITZ_RANDOM_N10_000,0.0154514171,markowitz,1.0154514171
1400,2015-02-01,RANDOM_N10_000,MARKOWITZ_RANDOM_N10_000,0.0123849553,markowitz,1.0280277376
2600,2015-03-01,RANDOM_N10_000,MARKOWITZ_RANDOM_N10_000,-0.0093977034,markowitz,1.0183666378
3800,2015-04-01,RANDOM_N10_000,MARKOWITZ_RANDOM_N10_000,0.0515067178,markowitz,1.0708193608
5000,2015-05-01,RANDOM_N10_000,MARKOWITZ_RANDOM_N10_000,0.0619350658,markowitz,1.1371406283



Markowitz concluído com sucesso.
CPU times: total: 42min 35s
Wall time: 39min


# 7) Simulação

In [9]:
# ==========================================================
# ETAPA 7 - SIMULAÇÃO
# ==========================================================
"""
Descrição:
    Calcula retornos mensais de todas as carteiras, consolida a base final
    e cria séries-resumo para visualização no HTML.

Processo:
    - Simula:
        - aleatórias
        - setor
        - subsetor
        - markowitz
        - ibovespa
    - Cria carteira_exibicao com nomes mais legíveis
    - Alinha o período ao início do Markowitz
    - Calcula retorno acumulado
    - Cria séries-resumo por grupo para visualização:
        - aleatórias por tamanho (média, mediana, p10, p90)
        - markowitz por tamanho (média, mediana, p10, p90)
        - setor agregado (média, mediana, p10, p90)
        - subsetor agregado (média, mediana, p10, p90)

Retorno:
    - df_retornos_carteiras
    - df_retornos_series_resumo
"""

print("="*80)
print("ETAPA 7 - SIMULAÇÃO")
print("="*80)

# =========================
# BASE
# =========================
df_pivot = df_retornos_elegiveis.pivot(
    index="data_ref",
    columns="ticker",
    values="retorno_mensal"
).sort_index()

# =========================
# FUNÇÃO AUXILIAR
# =========================
def gerar_nome_exibicao(nome, tipo, n_ativos=np.nan):
    nome = str(nome)

    if tipo == "aleatoria":
        match = re.search(r"RANDOM_N(\d+)_(\d+)", nome)
        if match:
            n = match.group(1)
            idx = match.group(2)
            return f"Aleatória N{n} | {idx}"
        return f"Aleatória | {nome}"

    if tipo == "markowitz":
        match = re.search(r"RANDOM_N(\d+)_(\d+)", nome)
        if match:
            n = match.group(1)
            idx = match.group(2)
            return f"Markowitz N{n} | {idx}"
        return f"Markowitz | {nome}"

    if tipo == "setor":
        nome_fmt = nome.replace("SETOR_", "").replace("_", " ")
        return f"Setor | {nome_fmt}"

    if tipo == "subsetor":
        nome_fmt = nome.replace("SUBSETOR_", "").replace("_", " ")
        return f"Subsetor | {nome_fmt}"

    if tipo == "benchmark":
        return "Ibovespa"

    return nome

# =========================
# FUNÇÃO DE SIMULAÇÃO
# =========================
def simular(pesos_dict, tipo):

    res = []

    for nome, info in pesos_dict.items():

        ativos = info["ativos"]
        pesos = info["pesos"]

        df_sub = df_pivot[ativos].copy()

        # remove meses totalmente vazios
        df_sub = df_sub.dropna(how="all")

        # faltante = 0 para manter consistência de série
        df_sub = df_sub.fillna(0)

        if df_sub.empty:
            continue

        ret = df_sub.dot(pesos)

        n_ativos = np.nan
        if "n_ativos" in info:
            n_ativos = info["n_ativos"]

        carteira_exibicao = gerar_nome_exibicao(nome, tipo, n_ativos)

        df_temp = pd.DataFrame({
            "data_ref": df_sub.index,
            "carteira": nome,
            "carteira_exibicao": carteira_exibicao,
            "retorno": ret,
            "tipo": tipo,
            "n_ativos": n_ativos
        })

        res.append(df_temp)

    if len(res) == 0:
        return pd.DataFrame(columns=[
            "data_ref", "carteira", "carteira_exibicao", "retorno", "tipo", "n_ativos"
        ])

    return pd.concat(res, ignore_index=True)

# =========================
# EXECUÇÃO
# =========================
print("Aleatórias...")
df_ret_random = simular(dicionario_carteiras_aleatorias, "aleatoria")

print("Setor...")
df_ret_setor = simular(dicionario_carteiras_setoriais, "setor")

print("Subsetor...")
df_ret_subsetor = simular(dicionario_carteiras_subsetoriais, "subsetor")

# =========================
# MARKOWITZ
# =========================
print("Markowitz...")

df_ret_markowitz = df_ret_markowitz.copy()

df_ret_markowitz["tipo"] = "markowitz"

# garante carteira_exibicao mais legível
df_ret_markowitz["n_ativos"] = pd.to_numeric(
    df_ret_markowitz["carteira"].astype(str).str.extract(r"RANDOM_N(\d+)_")[0],
    errors="coerce"
)

df_ret_markowitz["carteira_exibicao"] = df_ret_markowitz.apply(
    lambda x: gerar_nome_exibicao(x["carteira"], "markowitz", x["n_ativos"]),
    axis=1
)

df_ret_markowitz = df_ret_markowitz[[
    "data_ref", "carteira", "carteira_exibicao", "retorno", "tipo", "n_ativos"
]].copy()

# =========================
# IBOV
# =========================
print("Benchmark...")

df_ibov_sim = df_ibov_mensal[[
    "data_ref", "retorno_mensal_ibov"
]].copy()

df_ibov_sim = df_ibov_sim.rename(columns={
    "retorno_mensal_ibov": "retorno"
})

df_ibov_sim["carteira"] = "IBOV"
df_ibov_sim["carteira_exibicao"] = "Ibovespa"
df_ibov_sim["tipo"] = "benchmark"
df_ibov_sim["n_ativos"] = np.nan

# =========================
# JUNÇÃO
# =========================
df_retornos_carteiras = pd.concat([
    df_ret_random,
    df_ret_setor,
    df_ret_subsetor,
    df_ret_markowitz,
    df_ibov_sim
], ignore_index=True)

# =========================
# ALINHAR PERÍODO
# =========================
data_inicio = pd.to_datetime(df_ret_markowitz["data_ref"].min())

df_retornos_carteiras = df_retornos_carteiras[
    df_retornos_carteiras["data_ref"] >= data_inicio
].copy()

# =========================
# GARANTE N_ATIVOS DAS ALEATÓRIAS
# =========================
mask_aleat = df_retornos_carteiras["tipo"] == "aleatoria"
if mask_aleat.any():
    df_retornos_carteiras.loc[mask_aleat, "n_ativos"] = pd.to_numeric(
        df_retornos_carteiras.loc[mask_aleat, "carteira"].astype(str).str.extract(r"RANDOM_N(\d+)_")[0],
        errors="coerce"
    )

# =========================
# RETORNO ACUMULADO
# =========================
df_retornos_carteiras = df_retornos_carteiras.sort_values(
    ["tipo", "carteira", "data_ref"]
)

df_retornos_carteiras["retorno_acum"] = (
    df_retornos_carteiras
    .groupby(["carteira", "tipo"])["retorno"]
    .transform(lambda x: (1 + x).cumprod())
)

# =========================
# SÉRIES-RESUMO PARA VISUALIZAÇÃO
# =========================
print("Criando séries-resumo...")

registros_resumo = []

def adicionar_resumos(df_base, nome_grupo, coluna_grupo=None):
    """
    coluna_grupo:
        - None  -> agrega tudo junto
        - str   -> agrega por grupo dentro da coluna
    """
    if df_base.empty:
        return

    if coluna_grupo is None:
        grupos = [("GERAL", df_base.copy())]
    else:
        grupos = list(df_base.groupby(coluna_grupo))

    for grupo_nome, df_grupo in grupos:

        pivot = df_grupo.pivot_table(
            index="data_ref",
            columns="carteira_exibicao",
            values="retorno_acum",
            aggfunc="last"
        ).sort_index()

        if pivot.empty:
            continue

        estatisticas = {
            "MEDIA": pivot.mean(axis=1),
            "MEDIANA": pivot.median(axis=1),
            "P10": pivot.quantile(0.10, axis=1),
            "P90": pivot.quantile(0.90, axis=1),
        }

        for nome_est, serie in estatisticas.items():

            if coluna_grupo is None:
                carteira = f"{nome_grupo}_{nome_est}"
                subgrupo_resumo = None
            else:
                grupo_fmt = str(grupo_nome).upper()

                if grupo_fmt.startswith("SETOR_"):
                    grupo_fmt = grupo_fmt[len("SETOR_"):]
                if grupo_fmt.startswith("SUBSETOR_"):
                    grupo_fmt = grupo_fmt[len("SUBSETOR_"):]

                try:
                    grupo_num = float(grupo_fmt)
                    if grupo_num.is_integer():
                        grupo_fmt = str(int(grupo_num))
                except:
                    pass

                carteira = f"{nome_grupo}_{grupo_fmt}_{nome_est}"
                subgrupo_resumo = grupo_fmt

            carteira = str(carteira).upper().replace(".0_", "_").replace(".0", "")

            ret_acum = serie.values
            ret_mensal = pd.Series(ret_acum).pct_change().fillna(0).values

            df_tmp = pd.DataFrame({
                "data_ref": serie.index,
                "carteira": carteira,
                "carteira_exibicao": carteira,
                "retorno": ret_mensal,
                "tipo": "resumo",
                "n_ativos": np.nan if coluna_grupo is None else pd.to_numeric(pd.Series([grupo_nome]), errors="coerce").iloc[0],
                "retorno_acum": ret_acum,
                "grupo_resumo": nome_grupo,
                "subgrupo_resumo": subgrupo_resumo,
                "estatistica_resumo": nome_est
            })

            registros_resumo.append(df_tmp)

# aleatórias por tamanho
df_aleat = df_retornos_carteiras[df_retornos_carteiras["tipo"] == "aleatoria"].copy()
if not df_aleat.empty:
    adicionar_resumos(df_aleat, "ALEATORIA", coluna_grupo="n_ativos")

# markowitz por tamanho
df_mark = df_retornos_carteiras[df_retornos_carteiras["tipo"] == "markowitz"].copy()
if not df_mark.empty:
    adicionar_resumos(df_mark, "MARKOWITZ", coluna_grupo="n_ativos")

# setor agregado
df_set = df_retornos_carteiras[df_retornos_carteiras["tipo"] == "setor"].copy()
if not df_set.empty:
    adicionar_resumos(df_set, "SETOR", coluna_grupo=None)

# subsetor agregado
df_sub = df_retornos_carteiras[df_retornos_carteiras["tipo"] == "subsetor"].copy()
if not df_sub.empty:
    adicionar_resumos(df_sub, "SUBSETOR", coluna_grupo=None)

# benchmark
df_ibov_resumo = df_retornos_carteiras[df_retornos_carteiras["tipo"] == "benchmark"].copy()
if not df_ibov_resumo.empty:
    df_ibov_resumo = df_ibov_resumo.copy()
    df_ibov_resumo["grupo_resumo"] = "BENCHMARK"
    df_ibov_resumo["subgrupo_resumo"] = "IBOV"
    df_ibov_resumo["estatistica_resumo"] = "SERIE"
    registros_resumo.append(df_ibov_resumo)

if len(registros_resumo) > 0:
    df_retornos_series_resumo = pd.concat(registros_resumo, ignore_index=True)
else:
    df_retornos_series_resumo = pd.DataFrame(columns=[
        "data_ref", "carteira", "carteira_exibicao", "retorno", "tipo",
        "n_ativos", "retorno_acum", "grupo_resumo", "subgrupo_resumo", "estatistica_resumo"
    ])

# =========================
# VALIDAÇÃO
# =========================
print("\nCarteiras por tipo:")
print(df_retornos_carteiras.groupby("tipo")["carteira"].nunique())

print("\nResumo séries-resumo:")
if not df_retornos_series_resumo.empty:
    print(df_retornos_series_resumo.groupby(["grupo_resumo", "subgrupo_resumo", "estatistica_resumo"]).size().head(20))

print("\nShape df_retornos_carteiras:")
print(df_retornos_carteiras.shape)

print("\nShape df_retornos_series_resumo:")
print(df_retornos_series_resumo.shape)

print("\nAmostra df_retornos_carteiras:")
display(df_retornos_carteiras.head())

print("\nAmostra df_retornos_series_resumo:")
display(df_retornos_series_resumo.head())

# =========================
# SALVAR
# =========================
df_retornos_carteiras.to_parquet(
    "resultados/dados_intermediarios/df_retornos_carteiras.parquet",
    index=False
)

df_retornos_series_resumo.to_parquet(
    "resultados/dados_intermediarios/df_retornos_series_resumo.parquet",
    index=False
)

print("\nSimulação concluída com sucesso.")

ETAPA 7 - SIMULAÇÃO
Aleatórias...
Setor...
Subsetor...
Markowitz...
Benchmark...
Criando séries-resumo...

Carteiras por tipo:
tipo
aleatoria    1200
benchmark       1
markowitz    1200
setor          10
subsetor       24
Name: carteira, dtype: int64

Resumo séries-resumo:
grupo_resumo  subgrupo_resumo  estatistica_resumo
ALEATORIA     10               MEDIA                 132
                               MEDIANA               132
                               P10                   132
                               P90                   132
              15               MEDIA                 132
                               MEDIANA               132
                               P10                   132
                               P90                   132
              20               MEDIA                 132
                               MEDIANA               132
                               P10                   132
                               P90               

,data_ref,carteira,carteira_exibicao,retorno,tipo,n_ativos,retorno_acum
38211,2015-01-01,RANDOM_N10_000,Aleatória N10 | 000,-0.0817866907,aleatoria,10.0,0.9182133093
38212,2015-02-01,RANDOM_N10_000,Aleatória N10 | 000,0.0895429571,aleatoria,10.0,1.0004328442
38213,2015-03-01,RANDOM_N10_000,Aleatória N10 | 000,0.0017292389,aleatoria,10.0,1.0021628316
38214,2015-04-01,RANDOM_N10_000,Aleatória N10 | 000,0.0695325041,aleatoria,10.0,1.0718457229
38215,2015-05-01,RANDOM_N10_000,Aleatória N10 | 000,0.1034293849,aleatoria,10.0,1.1827060667



Amostra df_retornos_series_resumo:


,data_ref,carteira,carteira_exibicao,retorno,tipo,n_ativos,retorno_acum,grupo_resumo,subgrupo_resumo,estatistica_resumo
0,2015-01-01,ALEATORIA_5_MEDIA,ALEATORIA_5_MEDIA,0.0000000000,resumo,5.0,0.9627346995,ALEATORIA,5,MEDIA
1,2015-02-01,ALEATORIA_5_MEDIA,ALEATORIA_5_MEDIA,0.0230044544,resumo,5.0,0.9848818859,ALEATORIA,5,MEDIA
2,2015-03-01,ALEATORIA_5_MEDIA,ALEATORIA_5_MEDIA,-0.0073550801,resumo,5.0,0.9776380008,ALEATORIA,5,MEDIA
3,2015-04-01,ALEATORIA_5_MEDIA,ALEATORIA_5_MEDIA,0.0513581103,resumo,5.0,1.0278476411,ALEATORIA,5,MEDIA
4,2015-05-01,ALEATORIA_5_MEDIA,ALEATORIA_5_MEDIA,-0.0075675969,resumo,5.0,1.0200693045,ALEATORIA,5,MEDIA



Simulação concluída com sucesso.


# 8) Métricas

In [10]:
# ==========================================================
# ETAPA 8 - MÉTRICAS
# ==========================================================
"""
Descrição:
    Calcula métricas de desempenho das carteiras e também métricas
    agregadas para as séries-resumo usadas na visualização.

Processo:
    - Calcula métricas por carteira e tipo
    - Mantém separação correta entre aleatoria e markowitz
    - Usa carteira_exibicao para apresentação
    - Preserva n_ativos quando já existir na base
    - Calcula métricas também para df_retornos_series_resumo

Retorno:
    - df_metricas_carteiras
    - df_metricas_series_resumo
"""

print("="*80)
print("ETAPA 8 - MÉTRICAS")
print("="*80)

df = df_retornos_carteiras.copy()
df_res = df_retornos_series_resumo.copy() if "df_retornos_series_resumo" in globals() else pd.DataFrame()

# =========================
# FUNÇÕES
# =========================
def max_drawdown(series):
    cum = (1 + series).cumprod()
    peak = cum.cummax()
    dd = (cum / peak) - 1
    return dd.min()

def downside_vol(series):
    neg = series[series < 0]
    if len(neg) == 0:
        return 0.0
    return np.std(neg, ddof=1) if len(neg) > 1 else 0.0

def obter_n_ativos(df_carteira, carteira, tipo, grupo_resumo=np.nan, subgrupo_resumo=np.nan):
    """
    Descrição:
        Obtém n_ativos priorizando a própria base. Se não existir,
        tenta inferir pelo nome da carteira.
    """

    if "n_ativos" in df_carteira.columns:
        valor = pd.to_numeric(df_carteira["n_ativos"], errors="coerce").dropna()
        if not valor.empty:
            return float(valor.iloc[0])

    if tipo in ["aleatoria", "markowitz"]:
        match = re.search(r"N(\d+)", str(carteira))
        if match:
            return float(match.group(1))

    if pd.notna(subgrupo_resumo):
        try:
            return float(subgrupo_resumo)
        except:
            pass

    return np.nan

def calcular_metricas_df(df_base, usar_grupo_resumo=False):
    metricas = []

    if df_base.empty:
        return pd.DataFrame()

    if usar_grupo_resumo:
        agrupador = ["carteira", "tipo", "carteira_exibicao", "grupo_resumo", "subgrupo_resumo", "estatistica_resumo"]
    else:
        agrupador = ["carteira", "tipo", "carteira_exibicao"]

    for chaves, df_carteira in df_base.groupby(agrupador):

        if usar_grupo_resumo:
            carteira, tipo, carteira_exibicao, grupo_resumo, subgrupo_resumo, estatistica_resumo = chaves
        else:
            carteira, tipo, carteira_exibicao = chaves
            grupo_resumo = np.nan
            subgrupo_resumo = np.nan
            estatistica_resumo = np.nan

        df_carteira = df_carteira.sort_values("data_ref").copy()
        ret = pd.to_numeric(df_carteira["retorno"], errors="coerce").dropna()

        n = len(ret)

        if n < 12:
            continue

        retorno_total = (1 + ret).prod() - 1
        retorno_anual = (1 + retorno_total) ** (12 / n) - 1 if n > 0 else np.nan

        vol = ret.std(ddof=1) * np.sqrt(12) if n > 1 else np.nan
        down_vol = downside_vol(ret) * np.sqrt(12)

        sharpe = retorno_anual / vol if pd.notna(vol) and vol not in [0, 0.0] else np.nan
        sortino = retorno_anual / down_vol if pd.notna(down_vol) and down_vol not in [0, 0.0] else np.nan

        mdd = max_drawdown(ret)
        calmar = retorno_anual / abs(mdd) if pd.notna(mdd) and mdd not in [0, 0.0] else np.nan

        data_inicio = df_carteira["data_ref"].min()
        data_fim = df_carteira["data_ref"].max()

        n_ativos = obter_n_ativos(
            df_carteira=df_carteira,
            carteira=carteira,
            tipo=tipo,
            grupo_resumo=grupo_resumo,
            subgrupo_resumo=subgrupo_resumo
        )

        metricas.append({
            "carteira": carteira,
            "carteira_exibicao": carteira_exibicao,
            "tipo": tipo,
            "grupo_resumo": grupo_resumo,
            "subgrupo_resumo": subgrupo_resumo,
            "estatistica_resumo": estatistica_resumo,
            "n_obs": n,
            "data_inicio": data_inicio,
            "data_fim": data_fim,
            "n_ativos": n_ativos,
            "retorno_total": retorno_total,
            "retorno_anual": retorno_anual,
            "volatilidade": vol,
            "downside_vol": down_vol,
            "sharpe": sharpe,
            "sortino": sortino,
            "max_drawdown": mdd,
            "calmar": calmar
        })

    return pd.DataFrame(metricas)

# =========================
# MÉTRICAS POR CARTEIRA
# =========================
print("Calculando métricas por carteira...")
df_metricas_carteiras = calcular_metricas_df(df, usar_grupo_resumo=False)

# =========================
# MÉTRICAS DAS SÉRIES-RESUMO
# =========================
print("Calculando métricas das séries-resumo...")
if not df_res.empty:
    df_res = df_res.copy().sort_values(["carteira", "tipo", "data_ref"])

    if "retorno" not in df_res.columns:
        df_res["retorno"] = np.nan

    mask_sem_retorno = df_res["retorno"].isna()

    if mask_sem_retorno.any():
        df_res["retorno"] = (
            df_res
            .groupby(["carteira", "tipo"])["retorno_acum"]
            .transform(lambda x: x / x.shift(1) - 1)
        )

        primeiro_idx = (
            df_res
            .groupby(["carteira", "tipo"])
            .head(1)
            .index
        )
        df_res.loc[primeiro_idx, "retorno"] = df_res.loc[primeiro_idx, "retorno_acum"] - 1

    df_metricas_series_resumo = calcular_metricas_df(df_res, usar_grupo_resumo=True)
else:
    df_metricas_series_resumo = pd.DataFrame()

# =========================
# VALIDAÇÃO
# =========================
print("\nResumo df_metricas_carteiras:")
print(df_metricas_carteiras["tipo"].value_counts())

if not df_metricas_series_resumo.empty:
    print("\nResumo df_metricas_series_resumo:")
    print(df_metricas_series_resumo["grupo_resumo"].value_counts(dropna=False))

print("\nAmostra df_metricas_carteiras:")
display(df_metricas_carteiras.head())

if not df_metricas_series_resumo.empty:
    print("\nAmostra df_metricas_series_resumo:")
    display(df_metricas_series_resumo.head())

# =========================
# SALVAR
# =========================
df_metricas_carteiras.to_parquet(
    "resultados/tabelas/df_metricas_carteiras.parquet",
    index=False
)

df_metricas_carteiras.to_parquet(
    "resultados/dados_intermediarios/df_metricas_carteiras.parquet",
    index=False
)

if not df_metricas_series_resumo.empty:
    df_metricas_series_resumo.to_parquet(
        "resultados/tabelas/df_metricas_series_resumo.parquet",
        index=False
    )

    df_metricas_series_resumo.to_parquet(
        "resultados/dados_intermediarios/df_metricas_series_resumo.parquet",
        index=False
    )

print("\nMétricas calculadas com sucesso.")

ETAPA 8 - MÉTRICAS
Calculando métricas por carteira...
Calculando métricas das séries-resumo...

Resumo df_metricas_carteiras:
tipo
aleatoria    1200
markowitz    1200
subsetor       24
setor          10
benchmark       1
Name: count, dtype: int64

Resumo df_metricas_series_resumo:
grupo_resumo
ALEATORIA    24
MARKOWITZ    24
BENCHMARK     1
Name: count, dtype: int64

Amostra df_metricas_carteiras:


,carteira,carteira_exibicao,tipo,grupo_resumo,subgrupo_resumo,estatistica_resumo,n_obs,data_inicio,data_fim,n_ativos,retorno_total,retorno_anual,volatilidade,downside_vol,sharpe,sortino,max_drawdown,calmar
0,IBOV,Ibovespa,benchmark,NaN,NaN,NaN,136,2015-01-01,2026-04-01,NaN,2.9459275701,0.1287594311,0.2163444777,0.1538714560,0.5951593147,0.8367986789,-0.3703218240,0.3476960383
1,RANDOM_N10_000,Aleatória N10 | 000,aleatoria,NaN,NaN,NaN,132,2015-01-01,2025-12-01,10.0,1.0868032992,0.0691626384,0.1913655887,0.1188448345,0.3614162761,0.5819574635,-0.3492944876,0.1980066703
2,RANDOM_N10_000,Markowitz N10 | 000,markowitz,NaN,NaN,NaN,132,2015-01-01,2025-12-01,10.0,1.8517801097,0.0999531431,0.1069088065,0.0724248172,0.9349383495,1.3800952073,-0.1519921114,0.6576205977
3,RANDOM_N10_001,Aleatória N10 | 001,aleatoria,NaN,NaN,NaN,132,2015-01-01,2025-12-01,10.0,2.9475274194,0.1329516539,0.2239879915,0.1312202701,0.5935659901,1.0131944847,-0.3205536341,0.4147563459
4,RANDOM_N10_001,Markowitz N10 | 001,markowitz,NaN,NaN,NaN,132,2015-01-01,2025-12-01,10.0,2.5527928730,0.1221523107,0.1598942723,0.0793245259,0.7639567631,1.5399059668,-0.2703804275,0.4517794126



Amostra df_metricas_series_resumo:


,carteira,carteira_exibicao,tipo,grupo_resumo,subgrupo_resumo,estatistica_resumo,n_obs,data_inicio,data_fim,n_ativos,retorno_total,retorno_anual,volatilidade,downside_vol,sharpe,sortino,max_drawdown,calmar
0,ALEATORIA_10_MEDIA,ALEATORIA_10_MEDIA,resumo,ALEATORIA,10,MEDIA,132,2015-01-01,2025-12-01,10.0,3.4934013825,0.1463705471,0.1842582722,0.1315257914,0.7943770740,1.1128657390,-0.2964611140,0.4937259565
1,ALEATORIA_10_MEDIANA,ALEATORIA_10_MEDIANA,resumo,ALEATORIA,10,MEDIANA,132,2015-01-01,2025-12-01,10.0,2.9736592097,0.1336314199,0.1801832447,0.1282943596,0.7416417666,1.0416001168,-0.3153887561,0.4237038174
2,ALEATORIA_10_P10,ALEATORIA_10_P10,resumo,ALEATORIA,10,P10,132,2015-01-01,2025-12-01,10.0,0.9377475170,0.0619838842,0.2079580452,0.1318569445,0.2980595633,0.4700843355,-0.3898804601,0.1589817662
3,ALEATORIA_10_P90,ALEATORIA_10_P90,resumo,ALEATORIA,10,P90,132,2015-01-01,2025-12-01,10.0,6.1121206314,0.1952382003,0.1989562975,0.1341806744,0.9813119905,1.4550396414,-0.3340098939,0.5845281948
4,ALEATORIA_15_MEDIA,ALEATORIA_15_MEDIA,resumo,ALEATORIA,15,MEDIA,132,2015-01-01,2025-12-01,15.0,3.7960264291,0.1531832466,0.1844486430,0.1333494472,0.8304926730,1.1487355202,-0.2967713329,0.5161659148



Métricas calculadas com sucesso.


# 9) Análise Estatística

In [11]:
# ==========================================================
# ETAPA 9 - ANÁLISE ESTATÍSTICA
# ==========================================================
"""
Descrição:
    Realiza testes estatísticos entre estratégias:
    - Mann-Whitney (não paramétrico, independente)
    - Bootstrap (diferença de médias)
    - Permutação (teste empírico)
    - Wilcoxon pareado (mesma carteira: aleatoria vs markowitz)

Retorno:
    - df_estatisticas
"""

print("="*80)
print("ETAPA 9 - ANÁLISE ESTATÍSTICA")
print("="*80)

# =========================
# BASE
# =========================
df = df_metricas_carteiras.copy()

# =========================
# RESUMO DISTRIBUIÇÕES
# =========================
resumo = (
    df.groupby("tipo", as_index=False)
    .agg(
        retorno_mean=("retorno_anual", "mean"),
        retorno_median=("retorno_anual", "median"),
        sharpe_mean=("sharpe", "mean"),
        sharpe_median=("sharpe", "median"),
        mdd_mean=("max_drawdown", "mean"),
        mdd_median=("max_drawdown", "median")
    )
)

print("\nResumo distribuições:")
display(resumo)

# =========================
# EXTRAÇÃO DOS GRUPOS
# =========================
ret_aleat = df[df["tipo"] == "aleatoria"]["retorno_anual"].dropna()
ret_setor = df[df["tipo"] == "setor"]["retorno_anual"].dropna()
ret_mark = df[df["tipo"] == "markowitz"]["retorno_anual"].dropna()

# =========================
# TESTES
# =========================
resultados = []

# -------------------------
# 1) Mann-Whitney
# -------------------------
if len(ret_aleat) > 0 and len(ret_setor) > 0:
    stat, p = mannwhitneyu(ret_aleat, ret_setor, alternative="two-sided")
    resultados.append({
        "teste": "Mann-Whitney",
        "comparacao": "aleatoria vs setor",
        "estatistica": stat,
        "p_valor": p,
        "ci_inf": np.nan,
        "ci_sup": np.nan
    })

if len(ret_mark) > 0 and len(ret_aleat) > 0:
    stat, p = mannwhitneyu(ret_mark, ret_aleat, alternative="two-sided")
    resultados.append({
        "teste": "Mann-Whitney",
        "comparacao": "markowitz vs aleatoria",
        "estatistica": stat,
        "p_valor": p,
        "ci_inf": np.nan,
        "ci_sup": np.nan
    })

# -------------------------
# 2) Bootstrap
# -------------------------
if len(ret_mark) > 0 and len(ret_aleat) > 0:

    n_boot = 5000
    diffs = []

    for _ in range(n_boot):
        sample_mark = np.random.choice(ret_mark, size=len(ret_mark), replace=True)
        sample_aleat = np.random.choice(ret_aleat, size=len(ret_aleat), replace=True)

        diffs.append(sample_mark.mean() - sample_aleat.mean())

    diffs = np.array(diffs)

    resultados.append({
        "teste": "Bootstrap",
        "comparacao": "markowitz - aleatoria",
        "estatistica": diffs.mean(),
        "p_valor": np.nan,
        "ci_inf": np.percentile(diffs, 2.5),
        "ci_sup": np.percentile(diffs, 97.5)
    })

# -------------------------
# 3) Permutação
# -------------------------
if len(ret_mark) > 0 and len(ret_aleat) > 0:

    n_perm = 2000
    combined = np.concatenate([ret_mark, ret_aleat])
    n_mark = len(ret_mark)

    obs_diff = ret_mark.mean() - ret_aleat.mean()
    count = 0

    for _ in range(n_perm):
        perm = np.random.permutation(combined)

        perm_mark = perm[:n_mark]
        perm_aleat = perm[n_mark:]

        diff = perm_mark.mean() - perm_aleat.mean()

        if abs(diff) >= abs(obs_diff):
            count += 1

    p_perm = count / n_perm

    resultados.append({
        "teste": "Permutacao",
        "comparacao": "markowitz vs aleatoria",
        "estatistica": obs_diff,
        "p_valor": p_perm,
        "ci_inf": np.nan,
        "ci_sup": np.nan
    })

# -------------------------
# 4) Wilcoxon pareado
# -------------------------
df_pair = df[df["tipo"].isin(["aleatoria", "markowitz"])].copy()

pivot = df_pair.pivot_table(
    index="carteira",
    columns="tipo",
    values="retorno_anual"
).dropna()

if not pivot.empty and "aleatoria" in pivot.columns and "markowitz" in pivot.columns:

    try:
        stat, p = wilcoxon(pivot["markowitz"], pivot["aleatoria"])
        resultados.append({
            "teste": "Wilcoxon (pareado)",
            "comparacao": "markowitz vs aleatoria (mesma carteira)",
            "estatistica": stat,
            "p_valor": p,
            "ci_inf": np.nan,
            "ci_sup": np.nan
        })
    except:
        pass

# =========================
# RESULTADO FINAL
# =========================
df_estatisticas = pd.DataFrame(resultados)

print("\nTestes estatísticos:")
display(df_estatisticas)

# =========================
# SALVAR
# =========================
df_estatisticas.to_parquet(
    "resultados/tabelas/df_estatisticas.parquet",
    index=False
)

df_estatisticas.to_parquet(
    "resultados/dados_intermediarios/df_estatisticas.parquet",
    index=False
)

print("\nAnálise estatística concluída com sucesso.")

ETAPA 9 - ANÁLISE ESTATÍSTICA

Resumo distribuições:


,tipo,retorno_mean,retorno_median,sharpe_mean,sharpe_median,mdd_mean,mdd_median
0,aleatoria,0.1328301653,0.1323039646,0.5527399648,0.5671964509,-0.3741035140,-0.3507263023
1,benchmark,0.1287594311,0.1287594311,0.5951593147,0.5951593147,-0.3703218240,-0.3703218240
2,markowitz,0.1244998497,0.1214470770,0.6569349896,0.6557454731,-0.3023055860,-0.2827671687
3,setor,0.1170117432,0.1223366341,0.5300373688,0.5202333450,-0.4272751692,-0.4082670269
4,subsetor,0.1178391174,0.1273610559,0.4627349907,0.4438493602,-0.4576318801,-0.4346733917



Testes estatísticos:


,teste,comparacao,estatistica,p_valor,ci_inf,ci_sup
0,Mann-Whitney,aleatoria vs setor,6818.0000000000,0.4575574181,NaN,NaN
1,Mann-Whitney,markowitz vs aleatoria,639196.0000000000,0.0000019321,NaN,NaN
2,Bootstrap,markowitz - aleatoria,-0.0083167351,NaN,-0.0130802085,-0.0036129207
3,Permutacao,markowitz vs aleatoria,-0.0083303156,0.0005000000,NaN,NaN
4,Wilcoxon (pareado),markowitz vs aleatoria (mesma carteira),291429.0000000000,0.0000000097,NaN,NaN



Análise estatística concluída com sucesso.


# 10) Gráficos Finais

In [12]:
# ==========================================================
# ETAPA 10 - GRÁFICOS
# ==========================================================
"""
Descrição:
    Gera os gráficos finais do projeto, priorizando séries-resumo
    para melhorar a leitura e evitar poluição visual.

Inclui:
    - Curva de capital principal
    - Curva de capital por tamanho das aleatórias
    - Distribuição de retorno anual
    - Distribuição de Sharpe
    - Boxplot de retorno anual
    - Boxplot de Sharpe
    - Drawdown principal
    - Drawdown por tamanho das aleatórias
    - Percentual que bate o IBOV
    - Dispersão risco vs retorno

Retorno:
    - PNGs salvos em resultados/graficos/
"""

print("="*80)
print("ETAPA 10 - GRÁFICOS")
print("="*80)

os.makedirs("resultados/graficos", exist_ok=True)

df = df_retornos_carteiras.copy()
df_m = df_metricas_carteiras.copy()
df_res = df_retornos_series_resumo.copy()

# =========================
# FUNÇÕES
# =========================
def calc_drawdown_from_acum(series):
    peak = series.cummax()
    return (series / peak) - 1

def salvar_fig(nome):
    plt.tight_layout()
    plt.savefig(f"resultados/graficos/{nome}", dpi=300, bbox_inches="tight")
    plt.close()

# =========================
# PREPARAÇÃO
# =========================
df["data_ref"] = pd.to_datetime(df["data_ref"], errors="coerce")
df_res["data_ref"] = pd.to_datetime(df_res["data_ref"], errors="coerce")

# benchmark
ibov = df[df["tipo"] == "benchmark"].sort_values("data_ref").copy()

# resumos úteis
res_media_aleat = df_res[
    (df_res["grupo_resumo"] == "ALEATORIA") &
    (df_res["estatistica_resumo"] == "MEDIA")
].copy()

res_mediana_aleat = df_res[
    (df_res["grupo_resumo"] == "ALEATORIA") &
    (df_res["estatistica_resumo"] == "MEDIANA")
].copy()

res_p10_aleat = df_res[
    (df_res["grupo_resumo"] == "ALEATORIA") &
    (df_res["estatistica_resumo"] == "P10")
].copy()

res_p90_aleat = df_res[
    (df_res["grupo_resumo"] == "ALEATORIA") &
    (df_res["estatistica_resumo"] == "P90")
].copy()

mark_media = df_res[
    (df_res["grupo_resumo"] == "MARKOWITZ") &
    (df_res["estatistica_resumo"] == "MEDIA")
].copy()

mark_mediana = df_res[
    (df_res["grupo_resumo"] == "MARKOWITZ") &
    (df_res["estatistica_resumo"] == "MEDIANA")
].copy()

setor_media = df_res[
    (df_res["grupo_resumo"] == "SETOR") &
    (df_res["estatistica_resumo"] == "MEDIA")
].copy()

subsetor_media = df_res[
    (df_res["grupo_resumo"] == "SUBSETOR") &
    (df_res["estatistica_resumo"] == "MEDIA")
].copy()

# =========================
# 1) CURVA DE CAPITAL PRINCIPAL
# =========================
print("Gerando curva de capital principal...")

plt.figure(figsize=(12, 6))

for n in sorted(res_media_aleat["subgrupo_resumo"].dropna().unique()):
    serie_media = res_media_aleat[res_media_aleat["subgrupo_resumo"] == n].sort_values("data_ref")
    serie_p10 = res_p10_aleat[res_p10_aleat["subgrupo_resumo"] == n].sort_values("data_ref")
    serie_p90 = res_p90_aleat[res_p90_aleat["subgrupo_resumo"] == n].sort_values("data_ref")

    plt.plot(serie_media["data_ref"], serie_media["retorno_acum"], label=f"Aleatória N{int(float(n))} média")
    plt.fill_between(
        serie_media["data_ref"],
        serie_p10["retorno_acum"].values,
        serie_p90["retorno_acum"].values,
        alpha=0.10
    )

if not mark_media.empty:
    plt.plot(mark_media["data_ref"], mark_media["retorno_acum"], label="Markowitz média", linewidth=2.2)

if not ibov.empty:
    plt.plot(ibov["data_ref"], ibov["retorno_acum"], label="IBOV", linewidth=2.2)

plt.title("Curva de Capital - Resumo das Estratégias")
plt.xlabel("Data")
plt.ylabel("Retorno acumulado")
plt.legend()
plt.grid(True)

salvar_fig("curva_capital.png")

# =========================
# 2) CURVA DE CAPITAL - ALEATÓRIAS POR TAMANHO
# =========================
print("Gerando curva de capital por tamanho...")

plt.figure(figsize=(12, 6))

for n in sorted(res_media_aleat["subgrupo_resumo"].dropna().unique()):
    serie = res_media_aleat[res_media_aleat["subgrupo_resumo"] == n].sort_values("data_ref")
    plt.plot(serie["data_ref"], serie["retorno_acum"], label=f"N{int(float(n))}")

if not ibov.empty:
    plt.plot(ibov["data_ref"], ibov["retorno_acum"], label="IBOV", linewidth=2.2)

plt.title("Curva de Capital - Aleatórias por Tamanho")
plt.xlabel("Data")
plt.ylabel("Retorno acumulado")
plt.legend()
plt.grid(True)

salvar_fig("curva_capital_aleatorias_tamanho.png")

# =========================
# 3) DISTRIBUIÇÃO DE RETORNO ANUAL
# =========================
print("Gerando distribuição de retorno anual...")

plt.figure(figsize=(10, 5))

for tipo in ["aleatoria", "markowitz", "setor", "subsetor"]:
    subset = df_m[df_m["tipo"] == tipo]["retorno_anual"].dropna()
    if len(subset) > 0:
        plt.hist(subset, bins=40, alpha=0.40, label=tipo)

ibov_ret = df_m.loc[df_m["tipo"] == "benchmark", "retorno_anual"]
if len(ibov_ret) > 0:
    plt.axvline(ibov_ret.iloc[0], linestyle="--", label="IBOV")

plt.title("Distribuição de Retorno Anual")
plt.xlabel("Retorno anual")
plt.ylabel("Frequência")
plt.legend()
plt.grid(True)

salvar_fig("dist_retorno.png")

# =========================
# 4) DISTRIBUIÇÃO DE SHARPE
# =========================
print("Gerando distribuição de Sharpe...")

plt.figure(figsize=(10, 5))

for tipo in ["aleatoria", "markowitz", "setor", "subsetor"]:
    subset = df_m[df_m["tipo"] == tipo]["sharpe"].dropna()
    if len(subset) > 0:
        plt.hist(subset, bins=40, alpha=0.40, label=tipo)

ibov_sharpe = df_m.loc[df_m["tipo"] == "benchmark", "sharpe"]
if len(ibov_sharpe) > 0:
    plt.axvline(ibov_sharpe.iloc[0], linestyle="--", label="IBOV")

plt.title("Distribuição de Sharpe")
plt.xlabel("Sharpe")
plt.ylabel("Frequência")
plt.legend()
plt.grid(True)

salvar_fig("dist_sharpe.png")

# =========================
# 5) BOXPLOT - RETORNO
# =========================
print("Gerando boxplot de retorno...")

dados = []
labels = []

for tipo, label in [
    ("aleatoria", "Aleatória"),
    ("markowitz", "Markowitz"),
    ("setor", "Setor"),
    ("subsetor", "Subsetor")
]:
    serie = df_m[df_m["tipo"] == tipo]["retorno_anual"].dropna()
    if len(serie) > 0:
        dados.append(serie)
        labels.append(label)

plt.figure(figsize=(9, 5))
plt.boxplot(dados, tick_labels=labels)
plt.title("Comparação de Retorno Anual")
plt.ylabel("Retorno anual")
plt.grid(True)

salvar_fig("boxplot_retorno.png")

# =========================
# 6) BOXPLOT - SHARPE
# =========================
print("Gerando boxplot de Sharpe...")

dados = []
labels = []

for tipo, label in [
    ("aleatoria", "Aleatória"),
    ("markowitz", "Markowitz"),
    ("setor", "Setor"),
    ("subsetor", "Subsetor")
]:
    serie = df_m[df_m["tipo"] == tipo]["sharpe"].dropna()
    if len(serie) > 0:
        dados.append(serie)
        labels.append(label)

plt.figure(figsize=(9, 5))
plt.boxplot(dados, tick_labels=labels)
plt.title("Comparação de Sharpe")
plt.ylabel("Sharpe")
plt.grid(True)

salvar_fig("boxplot_sharpe.png")

# =========================
# 7) DRAWDOWN PRINCIPAL
# =========================
print("Gerando drawdown principal...")

plt.figure(figsize=(12, 6))

for n in sorted(res_media_aleat["subgrupo_resumo"].dropna().unique()):
    serie = res_media_aleat[res_media_aleat["subgrupo_resumo"] == n].sort_values("data_ref")
    dd = calc_drawdown_from_acum(serie["retorno_acum"])
    plt.plot(serie["data_ref"], dd, label=f"Aleatória N{int(float(n))} média")

if not mark_media.empty:
    dd_mark = calc_drawdown_from_acum(mark_media.sort_values("data_ref")["retorno_acum"])
    plt.plot(mark_media.sort_values("data_ref")["data_ref"], dd_mark, label="Markowitz média", linewidth=2.2)

if not ibov.empty:
    dd_ibov = calc_drawdown_from_acum(ibov.sort_values("data_ref")["retorno_acum"])
    plt.plot(ibov.sort_values("data_ref")["data_ref"], dd_ibov, label="IBOV", linewidth=2.2)

plt.title("Drawdown - Resumo das Estratégias")
plt.xlabel("Data")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)

salvar_fig("drawdown.png")

# =========================
# 8) DRAWDOWN - ALEATÓRIAS POR TAMANHO
# =========================
print("Gerando drawdown por tamanho...")

plt.figure(figsize=(12, 6))

for n in sorted(res_media_aleat["subgrupo_resumo"].dropna().unique()):
    serie = res_media_aleat[res_media_aleat["subgrupo_resumo"] == n].sort_values("data_ref")
    dd = calc_drawdown_from_acum(serie["retorno_acum"])
    plt.plot(serie["data_ref"], dd, label=f"N{int(float(n))}")

if not ibov.empty:
    dd_ibov = calc_drawdown_from_acum(ibov.sort_values("data_ref")["retorno_acum"])
    plt.plot(ibov.sort_values("data_ref")["data_ref"], dd_ibov, label="IBOV", linewidth=2.2)

plt.title("Drawdown - Aleatórias por Tamanho")
plt.xlabel("Data")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)

salvar_fig("drawdown_aleatorias_tamanho.png")

# =========================
# 9) % QUE BATEM O IBOV
# =========================
print("Gerando percentual que bate o IBOV...")

ibov_ret = df_m[df_m["tipo"] == "benchmark"]["retorno_anual"].iloc[0]

df_comp = df_m[df_m["tipo"] != "benchmark"].copy()
df_comp["bate_ibov"] = df_comp["retorno_anual"] > ibov_ret

res = df_comp.groupby("tipo")["bate_ibov"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(res.index, res.values)
plt.title("% de carteiras que batem o IBOV")
plt.ylabel("Percentual")
plt.grid(True)

salvar_fig("pct_bate_ibov.png")

# =========================
# 10) DISPERSÃO RISCO VS RETORNO
# =========================
print("Gerando dispersão risco vs retorno...")

plt.figure(figsize=(10, 6))

for tipo in ["aleatoria", "markowitz", "setor", "subsetor"]:
    tmp = df_m[df_m["tipo"] == tipo].copy()
    if not tmp.empty:
        plt.scatter(
            tmp["volatilidade"],
            tmp["retorno_anual"],
            alpha=0.35,
            label=tipo
        )

tmp_ibov = df_m[df_m["tipo"] == "benchmark"]
if not tmp_ibov.empty:
    plt.scatter(
        tmp_ibov["volatilidade"],
        tmp_ibov["retorno_anual"],
        label="benchmark"
    )

plt.title("Risco vs Retorno")
plt.xlabel("Volatilidade")
plt.ylabel("Retorno anual")
plt.legend()
plt.grid(True)

salvar_fig("scatter_risco_retorno.png")

print("\nGráficos gerados com sucesso.")

ETAPA 10 - GRÁFICOS
Gerando curva de capital principal...
Gerando curva de capital por tamanho...
Gerando distribuição de retorno anual...
Gerando distribuição de Sharpe...
Gerando boxplot de retorno...
Gerando boxplot de Sharpe...
Gerando drawdown principal...
Gerando drawdown por tamanho...
Gerando percentual que bate o IBOV...
Gerando dispersão risco vs retorno...

Gráficos gerados com sucesso.


# 11) HTML Final

In [13]:
# ==========================================================
# ETAPA 11 - HTML FINAL
# ==========================================================
"""
Descrição:
    Gera o HTML final interativo do projeto.

Inclui:
    - Seções organizadas para apresentação
    - KPIs automáticos
    - Gráfico interativo com seleção de múltiplas carteiras
    - Tabela sincronizada com o gráfico
    - Ranking interativo por métrica
    - Tabelas grandes com scroll
    - Imagens com zoom
    - Dados pesados exportados em arquivos JS externos
    - Uso de séries-resumo por padrão
    - Corte visual no período comum entre as séries

Pré-requisitos:
    - df_retornos_carteiras
    - df_retornos_series_resumo
    - df_metricas_carteiras
    - resumo 
    - gráficos já salvos em resultados/graficos

Retorno:
    - index.html
    - resultados/html/index.html
"""

print("=" * 80)
print("ETAPA 11 - HTML FINAL")
print("=" * 80)

# ==========================================================
# 1) CONFIGURAÇÕES
# ==========================================================
SALVAR_HTML_NA_RAIZ = True

RESULTADOS_DIR = Path("resultados")
GRAFICOS_DIR = RESULTADOS_DIR / "graficos"
TABELAS_DIR = RESULTADOS_DIR / "tabelas"
HTML_DIR = RESULTADOS_DIR / "html"
HTML_ASSETS_DIR = RESULTADOS_DIR / "html_assets"

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)
TABELAS_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)
HTML_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

ARQUIVO_HTML_FINAL = Path("index.html") if SALVAR_HTML_NA_RAIZ else HTML_DIR / "index.html"

ARQ_JS_RETORNOS_PARTS = [
    HTML_ASSETS_DIR / "retornos_data_part1.js",
    HTML_ASSETS_DIR / "retornos_data_part2.js",
    HTML_ASSETS_DIR / "retornos_data_part3.js",
]

ARQ_JS_METRICAS = HTML_ASSETS_DIR / "metricas_data.js"
ARQ_JS_TESTES = HTML_ASSETS_DIR / "testes_data.js"
ARQ_JS_RESUMO = HTML_ASSETS_DIR / "resumo_data.js"
ARQ_JS_META = HTML_ASSETS_DIR / "meta_data.js"

print(f"HTML final: {ARQUIVO_HTML_FINAL}")
print(f"Assets dir: {HTML_ASSETS_DIR}")

# ==========================================================
# 2) VALIDAÇÃO DAS BASES
# ==========================================================
if "df_retornos_carteiras" not in globals():
    raise ValueError("df_retornos_carteiras não está disponível.")

if "df_retornos_series_resumo" not in globals():
    raise ValueError("df_retornos_series_resumo não está disponível.")

if "df_metricas_carteiras" not in globals():
    raise ValueError("df_metricas_carteiras não está disponível.")

if "df_estatisticas" in globals():
    df_testes_html = df_estatisticas.copy()
elif "df_testes_estatisticos" in globals():
    df_testes_html = df_testes_estatisticos.copy()
else:
    raise ValueError("df_estatisticas ou df_testes_estatisticos não está disponível.")

if "resumo" in globals():
    df_resumo_html = resumo.copy()
else:
    raise ValueError("df_resumo_distribuicoes ou resumo não está disponível.")

df_retornos_html = df_retornos_carteiras.copy()
df_series_resumo_html = df_retornos_series_resumo.copy()
df_metricas_html = df_metricas_carteiras.copy()

# ==========================================================
# 3) PADRONIZAÇÃO DAS BASES
# ==========================================================
df_retornos_html["data_ref"] = pd.to_datetime(df_retornos_html["data_ref"], errors="coerce")
df_series_resumo_html["data_ref"] = pd.to_datetime(df_series_resumo_html["data_ref"], errors="coerce")

df_retornos_html = df_retornos_html.dropna(subset=["data_ref", "carteira", "tipo"]).copy()
df_series_resumo_html = df_series_resumo_html.dropna(subset=["data_ref", "carteira", "tipo"]).copy()

for col in ["retorno", "retorno_acum", "n_ativos"]:
    if col in df_retornos_html.columns:
        df_retornos_html[col] = pd.to_numeric(df_retornos_html[col], errors="coerce")

for col in ["retorno", "retorno_acum"]:
    if col in df_series_resumo_html.columns:
        df_series_resumo_html[col] = pd.to_numeric(df_series_resumo_html[col], errors="coerce")

colunas_metricas = [
    "retorno_total", "retorno_anual", "volatilidade", "downside_vol",
    "sharpe", "sortino", "max_drawdown", "calmar", "n_obs", "n_ativos"
]
for col in colunas_metricas:
    if col in df_metricas_html.columns:
        df_metricas_html[col] = pd.to_numeric(df_metricas_html[col], errors="coerce")

for col in ["data_inicio", "data_fim"]:
    if col in df_metricas_html.columns:
        df_metricas_html[col] = pd.to_datetime(df_metricas_html[col], errors="coerce")

# fallback carteira_exibicao
if "carteira_exibicao" not in df_retornos_html.columns:
    df_retornos_html["carteira_exibicao"] = df_retornos_html["carteira"].astype(str)

if "carteira_exibicao" not in df_series_resumo_html.columns:
    df_series_resumo_html["carteira_exibicao"] = df_series_resumo_html["carteira"].astype(str)

if "carteira_exibicao" not in df_metricas_html.columns:
    df_metricas_html["carteira_exibicao"] = df_metricas_html["carteira"].astype(str)
    
# ==========================================================
# 4) PERÍODO COMUM PARA VISUALIZAÇÃO
# ==========================================================
tipos_visual = ["benchmark", "aleatoria", "markowitz", "setor", "subsetor"]

datas_min = []
datas_max = []

for tipo in tipos_visual:
    tmp = df_retornos_html[df_retornos_html["tipo"] == tipo]
    if not tmp.empty:
        datas_min.append(tmp["data_ref"].min())
        datas_max.append(tmp["data_ref"].max())

if len(datas_min) == 0 or len(datas_max) == 0:
    raise ValueError("Não foi possível determinar o período comum das séries.")

DATA_INICIO_COMUM = max(datas_min)
DATA_FIM_COMUM = min(datas_max)

df_retornos_html = df_retornos_html[
    (df_retornos_html["data_ref"] >= DATA_INICIO_COMUM) &
    (df_retornos_html["data_ref"] <= DATA_FIM_COMUM)
].copy()

df_series_resumo_html = df_series_resumo_html[
    (df_series_resumo_html["data_ref"] >= DATA_INICIO_COMUM) &
    (df_series_resumo_html["data_ref"] <= DATA_FIM_COMUM)
].copy()

print(f"Período comum do HTML: {DATA_INICIO_COMUM.date()} até {DATA_FIM_COMUM.date()}")

# ==========================================================
# 5) FUNÇÕES AUXILIARES
# ==========================================================
def caminho_relativo_html(path_destino, path_html_final):
    return os.path.relpath(Path(path_destino), start=Path(path_html_final).parent).replace("\\", "/")

def format_pct(x, casas=1):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{casas}%}"

def format_num(x, casas=2):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{casas}f}"

def dataframe_para_html(df, table_id):
    if df is None or len(df) == 0:
        return f"<p>Nenhum dado disponível para <b>{html.escape(table_id)}</b>.</p>"

    df_html = df.copy()

    if isinstance(df_html.index, pd.Index) and df_html.index.name is not None:
        df_html = df_html.reset_index()

    for col in df_html.columns:
        if pd.api.types.is_datetime64_any_dtype(df_html[col]):
            df_html[col] = df_html[col].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_float_dtype(df_html[col]):
            df_html[col] = df_html[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
        else:
            df_html[col] = df_html[col].astype(str)

    return df_html.to_html(
        index=False,
        classes="display compact stripe nowrap",
        table_id=table_id,
        escape=False,
        border=0
    )

def montar_bloco_graficos(lista_graficos, titulo_secao=None):
    partes = []
    if titulo_secao:
        partes.append(f"<h3>{html.escape(titulo_secao)}</h3>")
    partes.append('<div class="graficos-grid">')

    for nome_arquivo, titulo in lista_graficos:
        path_img = GRAFICOS_DIR / nome_arquivo
        if path_img.exists():
            rel = caminho_relativo_html(path_img, ARQUIVO_HTML_FINAL)
            partes.append(f"""
            <div class="grafico-card">
                <h4>{html.escape(titulo)}</h4>
                <img src="{rel}" alt="{html.escape(titulo)}" class="grafico-img zoomable-img">
            </div>
            """)

    partes.append("</div>")
    return "\n".join(partes)

def extrair_n_aleatoria(nome_carteira):
    m = re.search(r"RANDOM_N(\d+)_", str(nome_carteira))
    return int(m.group(1)) if m else None

# ==========================================================
# 6) PREPARAÇÃO DAS BASES INTERATIVAS
# ==========================================================
print("Preparando bases interativas...")

# -------------------------
# 6.1) Séries individuais
# -------------------------
base_individual = df_retornos_html[[
    "data_ref", "carteira", "carteira_exibicao", "tipo", "retorno", "retorno_acum"
]].copy()

registros_plot_ind = []

for metrica in ["retorno", "retorno_acum"]:
    tmp = base_individual[["data_ref", "carteira", "carteira_exibicao", "tipo", metrica]].copy()
    tmp = tmp.rename(columns={metrica: "valor"})
    tmp["metrica"] = metrica
    tmp["modo_visual"] = "individual"
    registros_plot_ind.append(tmp)

df_plot_individual = pd.concat(registros_plot_ind, ignore_index=True)
df_plot_individual = df_plot_individual.dropna(subset=["valor"]).copy()

# -------------------------
# 6.2) Séries-resumo
# -------------------------
base_resumo = df_series_resumo_html.copy()

if "grupo_resumo" not in base_resumo.columns:
    base_resumo["grupo_resumo"] = np.nan
if "subgrupo_resumo" not in base_resumo.columns:
    base_resumo["subgrupo_resumo"] = np.nan
if "estatistica_resumo" not in base_resumo.columns:
    base_resumo["estatistica_resumo"] = np.nan

registros_plot_resumo = []

for metrica in ["retorno", "retorno_acum"]:
    if metrica not in base_resumo.columns:
        continue

    tmp = base_resumo[[
        "data_ref", "carteira", "carteira_exibicao", "tipo",
        "grupo_resumo", "subgrupo_resumo", "estatistica_resumo", metrica
    ]].copy()

    tmp = tmp.rename(columns={metrica: "valor"})
    tmp["metrica"] = metrica
    tmp["modo_visual"] = "resumo"
    registros_plot_resumo.append(tmp)

df_plot_resumo = pd.concat(registros_plot_resumo, ignore_index=True) if len(registros_plot_resumo) > 0 else pd.DataFrame()
if not df_plot_resumo.empty:
    df_plot_resumo = df_plot_resumo.dropna(subset=["valor"]).copy()

# -------------------------
# 6.3) Base combinada para o gráfico
# -------------------------
colunas_plot = [
    "data_ref", "carteira", "carteira_exibicao", "tipo", "metrica", "valor", "modo_visual",
    "grupo_resumo", "subgrupo_resumo", "estatistica_resumo"
]

for col in colunas_plot:
    if col not in df_plot_individual.columns:
        df_plot_individual[col] = np.nan
    if not df_plot_resumo.empty and col not in df_plot_resumo.columns:
        df_plot_resumo[col] = np.nan

df_plot_interativo = pd.concat([
    df_plot_resumo[colunas_plot],
    df_plot_individual[colunas_plot]
], ignore_index=True)

df_plot_interativo["data_ref"] = pd.to_datetime(df_plot_interativo["data_ref"], errors="coerce")
df_plot_interativo = df_plot_interativo.dropna(subset=["data_ref", "carteira_exibicao", "tipo", "metrica", "valor"]).copy()

# ==========================================================
# 7) TABELAS PARA O HTML
# ==========================================================
print("Preparando tabelas...")

df_resumo_tabela = df_resumo_html.copy()
if isinstance(df_resumo_tabela.index, pd.Index) and df_resumo_tabela.index.name is not None:
    df_resumo_tabela = df_resumo_tabela.reset_index()
elif "tipo" not in df_resumo_tabela.columns:
    df_resumo_tabela = df_resumo_tabela.reset_index().rename(columns={"index": "tipo"})

df_testes_tabela = df_testes_html.copy().reset_index(drop=True)
df_metricas_tabela = df_metricas_html.copy().sort_values(["tipo", "carteira_exibicao"]).reset_index(drop=True)

tabela_resumo_html = dataframe_para_html(df_resumo_tabela, "tabela_resumo")
tabela_testes_html = dataframe_para_html(df_testes_tabela, "tabela_testes")
tabela_metricas_html = dataframe_para_html(df_metricas_tabela, "tabela_metricas")

# ==========================================================
# 8) KPIs E LEITURA AUTOMÁTICA
# ==========================================================
print("Calculando KPIs e narrativa automática...")

df_resumo_aux = df_resumo_tabela.copy()

melhor_retorno = df_resumo_aux.sort_values("retorno_mean", ascending=False).iloc[0]
melhor_sharpe = df_resumo_aux.sort_values("sharpe_mean", ascending=False).iloc[0]
melhor_draw = df_resumo_aux.sort_values("mdd_mean", ascending=False).iloc[0]  # menos negativo

linha_perm = df_testes_tabela[df_testes_tabela["teste"].astype(str).str.contains("Permut", case=False, na=False)]
linha_boot = df_testes_tabela[df_testes_tabela["teste"].astype(str).str.contains("Bootstrap", case=False, na=False)]
linha_wil = df_testes_tabela[df_testes_tabela["teste"].astype(str).str.contains("Wilcoxon", case=False, na=False)]
linha_mw = df_testes_tabela[
    (df_testes_tabela["teste"].astype(str).str.contains("Mann", case=False, na=False)) &
    (df_testes_tabela["comparacao"].astype(str).str.contains("markowitz", case=False, na=False))
]

perm_p = float(linha_perm["p_valor"].iloc[0]) if len(linha_perm) > 0 and pd.notna(linha_perm["p_valor"].iloc[0]) else np.nan
boot_diff = float(linha_boot["estatistica"].iloc[0]) if len(linha_boot) > 0 and pd.notna(linha_boot["estatistica"].iloc[0]) else np.nan
boot_ci_inf = float(linha_boot["ci_inf"].iloc[0]) if len(linha_boot) > 0 and pd.notna(linha_boot["ci_inf"].iloc[0]) else np.nan
boot_ci_sup = float(linha_boot["ci_sup"].iloc[0]) if len(linha_boot) > 0 and pd.notna(linha_boot["ci_sup"].iloc[0]) else np.nan
wil_p = float(linha_wil["p_valor"].iloc[0]) if len(linha_wil) > 0 and pd.notna(linha_wil["p_valor"].iloc[0]) else np.nan
mw_p = float(linha_mw["p_valor"].iloc[0]) if len(linha_mw) > 0 and pd.notna(linha_mw["p_valor"].iloc[0]) else np.nan

ret_aleat = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "aleatoria", "retorno_mean"].iloc[0])
ret_mark = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "markowitz", "retorno_mean"].iloc[0])
ret_ibov = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "benchmark", "retorno_mean"].iloc[0])

sh_aleat = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "aleatoria", "sharpe_mean"].iloc[0])
sh_mark = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "markowitz", "sharpe_mean"].iloc[0])
sh_ibov = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "benchmark", "sharpe_mean"].iloc[0])

mdd_aleat = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "aleatoria", "mdd_mean"].iloc[0])
mdd_mark = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "markowitz", "mdd_mean"].iloc[0])
mdd_ibov = float(df_resumo_aux.loc[df_resumo_aux["tipo"] == "benchmark", "mdd_mean"].iloc[0])

n_por_tipo = df_metricas_tabela["tipo"].value_counts().to_dict()

leitura_mark_retorno = "menor" if ret_mark < ret_aleat else "maior"
leitura_boot = "negativa" if pd.notna(boot_diff) and boot_diff < 0 else "positiva"

takeaways = [
    (
        "Markowitz não maximizou retorno, mas maximizou eficiência",
        f"Seu retorno médio foi {leitura_mark_retorno} que o das aleatórias, mas o grupo entregou Sharpe médio de {format_num(sh_mark, 3)} e drawdown médio de {format_pct(mdd_mark)}."
    ),
    (
        "Aleatórias ficaram fortes em retorno",
        f"As carteiras aleatórias tiveram retorno anual médio de {format_pct(ret_aleat)}, acima do benchmark em {format_pct(ret_aleat - ret_ibov)}."
    ),
    (
        "Concentração continuou punida no risco",
        "Setores e subsetores concentrados ficaram atrás em Sharpe e exibiram drawdowns médios mais profundos."
    ),
    (
        "Métodos empíricos reforçaram a leitura do efeito",
        f"O bootstrap apontou diferença média {leitura_boot} de {format_pct(boot_diff)} para Markowitz vs aleatórias, com IC [{format_pct(boot_ci_inf)}, {format_pct(boot_ci_sup)}]."
    ),
]

cards_takeaways = "\n".join([
    f"""
    <div class="takeaway-card">
        <h3>{html.escape(titulo)}</h3>
        <p>{html.escape(texto)}</p>
    </div>
    """ for titulo, texto in takeaways
])

conteudo_resumo = f"""
<section class="executive-summary">
    <div class="exec-main">
        <h2>Resumo Executivo</h2>
        <p>
            Este projeto comparou carteiras aleatórias, carteiras setoriais, carteiras subsetoriais,
            o benchmark do Ibovespa e uma família de carteiras com otimização de pesos via Markowitz.
        </p>
        <p>
            O objetivo central não foi identificar apenas quem teve maior retorno, mas sim entender
            como a combinação entre retorno e risco muda quando saímos de uma diversificação simples
            para uma alocação otimizada.
        </p>

        <div class="highlight-grid">
            <div class="highlight-card">
                <span class="label">Melhor retorno médio</span>
                <div class="value">{html.escape(str(melhor_retorno["tipo"]))}</div>
                <div class="small">{format_pct(melhor_retorno["retorno_mean"])}</div>
            </div>
            <div class="highlight-card">
                <span class="label">Melhor Sharpe médio</span>
                <div class="value">{html.escape(str(melhor_sharpe["tipo"]))}</div>
                <div class="small">{format_num(melhor_sharpe["sharpe_mean"], 3)}</div>
            </div>
            <div class="highlight-card">
                <span class="label">Melhor drawdown médio</span>
                <div class="value">{html.escape(str(melhor_draw["tipo"]))}</div>
                <div class="small">{format_pct(melhor_draw["mdd_mean"])}</div>
            </div>
            <div class="highlight-card">
                <span class="label">Período comum da análise visual</span>
                <div class="value">{DATA_INICIO_COMUM.strftime("%Y-%m")} → {DATA_FIM_COMUM.strftime("%Y-%m")}</div>
                <div class="small">Todas as séries cortadas no mesmo intervalo</div>
            </div>
        </div>
    </div>

    <div class="exec-side">
        <h2>Key Takeaways</h2>
        <div class="takeaways">
            {cards_takeaways}
        </div>
    </div>
</section>
"""

conteudo_introducao = """
<div class="texto-livre">
    <p>
        O projeto investiga até que ponto diversificação, concentração setorial e otimização de pesos
        geram resultados diferentes no mercado brasileiro. Em vez de olhar apenas o retorno final,
        a análise foi construída com foco em eficiência risco-retorno, drawdown e robustez estatística.
    </p>

    <p>
        A leitura final é mais rica do que um ranking simples. Algumas abordagens se destacam em retorno,
        outras em eficiência e outras em resistência a quedas, o que torna a comparação mais próxima do
        problema real de alocação enfrentado por um investidor.
    </p>
</div>
"""

conteudo_metodologia = f"""
<div class="texto-livre">
    <p>
        A metodologia foi estruturada em etapas sequenciais:
    </p>

    <ul>
        <li>Construção do universo elegível de ativos e benchmark do Ibovespa;</li>
        <li>Geração de <b>1200 carteiras aleatórias</b> com diferentes tamanhos;</li>
        <li>Construção de <b>10 carteiras setoriais</b> e <b>24 carteiras subsetoriais</b>;</li>
        <li>Otimização de pesos por Markowitz com janela de estimação de <b>60 meses</b>, sem look-ahead;</li>
        <li>Simulação mensal e consolidação das métricas de desempenho;</li>
        <li>Aplicação de testes estatísticos não paramétricos e empíricos.</li>
    </ul>

    <p>
        O HTML final usa séries-resumo por padrão para melhorar a leitura visual,
        preservando as carteiras individuais como camada exploratória avançada.
    </p>
</div>
"""

conteudo_resultados = f"""
<div class="texto-livre">
    <p>
        O resultado mais interessante do projeto não é simplesmente “quem ganhou em retorno”,
        mas sim a diferença entre <b>retorno puro</b> e <b>retorno ajustado ao risco</b>.
    </p>

    <p>
        As carteiras aleatórias tiveram retorno anual médio de <b>{format_pct(ret_aleat)}</b>,
        acima do benchmark do Ibovespa em <b>{format_pct(ret_aleat - ret_ibov)}</b>.
        Isso reforça a força da diversificação simples: mesmo sem otimização sofisticada,
        muitas carteiras aleatórias já ficaram competitivas com o índice.
    </p>

    <p>
        Parte desse resultado também reflete o papel do acaso. Ao considerar um grande número
        de carteiras possíveis, é esperado que algumas combinações apresentem desempenho acima
        do benchmark. Isso não significa superioridade estrutural da aleatoriedade, mas sim que
        a diversificação ampla, combinada com variabilidade natural, pode gerar resultados
        competitivos em termos de retorno.
    </p>

    <p>
        O grupo Markowitz, por sua vez, ficou com retorno anual médio de <b>{format_pct(ret_mark)}</b>,
        abaixo do grupo aleatório em cerca de <b>{format_pct(ret_mark - ret_aleat)}</b>.
        Isso não invalida o modelo. Na verdade, reforça a lógica correta de interpretação:
        Markowitz <b>não foi desenhado para maximizar retorno isoladamente</b>,
        mas para melhorar a relação entre retorno e risco.
    </p>

    <p>
        E foi exatamente isso que apareceu. O Sharpe médio do Markowitz ficou em
        <b>{format_num(sh_mark, 3)}</b>, acima do Sharpe das aleatórias
        (<b>{format_num(sh_aleat, 3)}</b>) e do benchmark
        (<b>{format_num(sh_ibov, 3)}</b>). Além disso, seu drawdown médio de
        <b>{format_pct(mdd_mark)}</b> foi menos severo do que o das aleatórias
        (<b>{format_pct(mdd_aleat)}</b>) e melhor também do que o do próprio índice
        (<b>{format_pct(mdd_ibov)}</b>).
    </p>

    <p>
        Isso muda a narrativa do projeto de forma importante:
        <b>Markowitz não venceu em retorno, mas venceu em eficiência</b>.
        Em outras palavras, aceitou abrir mão de parte da rentabilidade média
        para reduzir risco e controlar melhor a profundidade das perdas.
    </p>

    <p>
        Já as carteiras setoriais e subsetoriais continuaram entregando a leitura clássica da concentração:
        menor Sharpe médio e drawdowns mais profundos. A concentração, portanto, não foi recompensada
        de forma convincente no agregado.
    </p>

    <p>
        Nos testes estatísticos, a comparação entre Markowitz e aleatórias mostrou um quadro interessante.
        O bootstrap apontou diferença média de <b>{format_pct(boot_diff)}</b>,
        com intervalo de confiança de <b>[{format_pct(boot_ci_inf)}, {format_pct(boot_ci_sup)}]</b>.
        A permutação produziu p-valor de <b>{format_num(perm_p, 4)}</b>,
        enquanto o Wilcoxon pareado ficou em <b>{format_num(wil_p, 4)}</b>.
        Isso mostra que a diferença existe, mas sua leitura depende muito da métrica observada:
        retorno médio aponta para um lado; eficiência risco-retorno, para outro.
    </p>
</div>
"""

conteudo_conclusao = f"""
<div class="conclusao">
    <h2>Conclusão</h2>

    <p>
        Este projeto mostrou que a escolha entre diversificação simples, concentração e otimização
        não pode ser reduzida a uma única métrica. A análise final deixa claro que
        <b>retorno puro e eficiência risco-retorno não apontaram exatamente para o mesmo vencedor</b>.
    </p>

    <p>
        As carteiras aleatórias exibiram retorno anual médio de <b>{format_pct(ret_aleat)}</b>,
        superando o benchmark do Ibovespa, que ficou em <b>{format_pct(ret_ibov)}</b>.
        Esse resultado é relevante porque reforça uma ideia importante:
        <b>a diversificação simples já é poderosa</b>. Mesmo sem uma camada sofisticada de otimização,
        um conjunto grande de carteiras aleatórias foi capaz de competir bem com o índice.
    </p>

    <p>
        É importante destacar que parte desse desempenho também está associada ao efeito do acaso.
        Ao gerar um grande conjunto de carteiras, algumas combinações naturalmente se destacam e
        superam o benchmark. Isso não implica que investir de forma aleatória seja a melhor estratégia,
        mas evidencia que a aleatoriedade, quando combinada com diversificação, pode produzir bons
        resultados dentro de um conjunto amplo de possibilidades.
    </p>

    <p>
        O Markowitz, por outro lado, terminou com retorno anual médio de <b>{format_pct(ret_mark)}</b>,
        abaixo das aleatórias em aproximadamente <b>{format_pct(ret_mark - ret_aleat)}</b>.
        À primeira vista, isso poderia parecer uma derrota do método. Mas essa leitura seria incompleta.
        O ponto central é que <b>Markowitz não foi concebido para maximizar retorno absoluto</b>,
        e sim para maximizar a relação entre risco e retorno.
    </p>

    <p>
        Sob essa ótica, ele funcionou bem. O Sharpe médio do grupo Markowitz foi de
        <b>{format_num(sh_mark, 3)}</b>, superior ao das aleatórias
        (<b>{format_num(sh_aleat, 3)}</b>) e ao do próprio benchmark
        (<b>{format_num(sh_ibov, 3)}</b>). Além disso, o drawdown médio do Markowitz
        ficou em <b>{format_pct(mdd_mark)}</b>, menos severo do que o drawdown das
        aleatórias (<b>{format_pct(mdd_aleat)}</b>) e também melhor do que o do Ibovespa
        (<b>{format_pct(mdd_ibov)}</b>).
    </p>

    <p>
        Esse ponto torna a conclusão mais interessante. Em vez de dizer que “uma estratégia venceu tudo”,
        o projeto sugere que existem <b>vantagens diferentes para critérios diferentes</b>.
        As aleatórias foram fortes em retorno. O Markowitz foi forte em eficiência e preservação de capital.
        O benchmark permaneceu competitivo. Já as carteiras concentradas por setor e subsetor foram,
        no agregado, as menos eficientes.
    </p>

    <p>
        A análise estatística também reforça essa nuance. O bootstrap apontou diferença média de
        <b>{format_pct(boot_diff)}</b> entre Markowitz e aleatórias, com intervalo de confiança
        <b>[{format_pct(boot_ci_inf)}, {format_pct(boot_ci_sup)}]</b>. A permutação trouxe p-valor de
        <b>{format_num(perm_p, 4)}</b>, enquanto o Wilcoxon pareado ficou em
        <b>{format_num(wil_p, 4)}</b>. Isso mostra que a conclusão depende muito da pergunta feita:
        <b>retorno médio isolado</b> favorece mais as aleatórias; <b>qualidade do retorno</b> favorece mais o Markowitz.
    </p>

    <p>
        Em termos econômicos, esse é um resultado bastante plausível. Uma carteira otimizada tende a sacrificar
        parte da agressividade para reduzir volatilidade e drawdown. Já uma carteira igualmente ponderada e bem
        diversificada pode capturar mais upside médio, mas também aceita sofrer mais nas quedas.
    </p>

    <p>
        As carteiras setoriais e subsetoriais, por sua vez, reforçam uma leitura clássica do mercado:
        <b>concentração cobra seu preço</b>. Mesmo quando o retorno médio não fica tão distante visualmente,
        a piora em Sharpe e a profundidade maior dos drawdowns tornam essas abordagens menos atraentes
        no agregado.
    </p>

    <p>
        Portanto, a principal leitura prática do projeto é:
        <b>não existe uma única carteira “melhor” em todos os critérios</b>.
        O investidor que busca maior retorno médio pode enxergar valor na diversificação simples.
        O investidor que prioriza eficiência e controle de perdas encontra uma justificativa clara para o Markowitz.
        E o investidor que concentra demais a carteira tende a assumir mais risco sem receber compensação proporcional.
    </p>

    <h3>Principais aprendizados</h3>
    <ul>
        <li>As <b>carteiras aleatórias</b> ficaram fortes em retorno e superaram o benchmark no agregado.</li>
        <li>Parte desse desempenho reflete o efeito do acaso em um conjunto amplo de carteiras, o que não implica superioridade da aleatoriedade como estratégia.</li>
        <li>O <b>Markowitz</b> não venceu em retorno médio, mas se destacou em <b>Sharpe</b> e <b>drawdown</b>.</li>
        <li>O método de Markowitz cumpriu bem seu papel: <b>melhorar a eficiência risco-retorno</b>, não necessariamente maximizar retorno bruto.</li>
        <li>As carteiras <b>setoriais</b> e <b>subsetoriais</b> continuaram sendo as mais frágeis em termos de drawdown e Sharpe.</li>
        <li>A <b>diversificação simples</b> já se mostrou uma estratégia muito competitiva.</li>
        <li>Os testes estatísticos mostraram que a leitura depende da métrica usada, o que torna a análise mais rica e mais realista.</li>
        <li>Na prática, a decisão entre aleatórias e Markowitz depende do objetivo do investidor: <b>mais retorno médio</b> versus <b>mais eficiência e controle de perdas</b>.</li>
    </ul>
</div>
"""

# ==========================================================
# 9) EXPORTAÇÃO DAS BASES PESADAS PARA JS EXTERNO
# ==========================================================
print("Exportando bases JS...")

df_plot_export = df_plot_interativo.copy()
df_plot_export["data_ref"] = df_plot_export["data_ref"].dt.strftime("%Y-%m-%d")

df_metricas_export = df_metricas_tabela.copy()
for col in df_metricas_export.columns:
    if pd.api.types.is_datetime64_any_dtype(df_metricas_export[col]):
        df_metricas_export[col] = df_metricas_export[col].dt.strftime("%Y-%m-%d")

df_testes_export = df_testes_tabela.copy()
df_resumo_export = df_resumo_tabela.copy()

meta_payload = {
    "n_por_tipo": n_por_tipo,
    "melhor_retorno_tipo": str(melhor_retorno["tipo"]),
    "melhor_retorno_valor": float(melhor_retorno["retorno_mean"]),
    "melhor_sharpe_tipo": str(melhor_sharpe["tipo"]),
    "melhor_sharpe_valor": float(melhor_sharpe["sharpe_mean"]),
    "melhor_draw_tipo": str(melhor_draw["tipo"]),
    "melhor_draw_valor": float(melhor_draw["mdd_mean"]),
    "perm_p": None if pd.isna(perm_p) else float(perm_p),
    "boot_diff": None if pd.isna(boot_diff) else float(boot_diff),
    "boot_ci_inf": None if pd.isna(boot_ci_inf) else float(boot_ci_inf),
    "boot_ci_sup": None if pd.isna(boot_ci_sup) else float(boot_ci_sup),
    "wil_p": None if pd.isna(wil_p) else float(wil_p),
    "mw_p": None if pd.isna(mw_p) else float(mw_p),
    "data_inicio_comum": DATA_INICIO_COMUM.strftime("%Y-%m-%d"),
    "data_fim_comum": DATA_FIM_COMUM.strftime("%Y-%m-%d"),
}

# ----------------------------------------------------------
# EXPORTAÇÃO DE RETORNOS EM PARTES PARA EVITAR LIMITE DO GITHUB
# ----------------------------------------------------------
registros_retornos = df_plot_export.to_dict(orient="records")
n_total = len(registros_retornos)
n_partes = len(ARQ_JS_RETORNOS_PARTS)

if n_total == 0:
    raise ValueError("df_plot_export ficou vazio. Não há dados para exportar.")

tam_parte = int(np.ceil(n_total / n_partes))

for i, arq_saida in enumerate(ARQ_JS_RETORNOS_PARTS, start=1):
    ini = (i - 1) * tam_parte
    fim = min(i * tam_parte, n_total)
    parte = registros_retornos[ini:fim]

    arq_saida.write_text(
        f"window.RETORNOS_DATA_PART{i} = " + json.dumps(parte, ensure_ascii=False) + ";",
        encoding="utf-8"
    )
    print(f"Parte {i} exportada: {arq_saida.name} | registros: {len(parte):,}")

ARQ_JS_METRICAS.write_text(
    "window.METRICAS_DATA = " + json.dumps(df_metricas_export.to_dict(orient="records"), ensure_ascii=False) + ";",
    encoding="utf-8"
)
ARQ_JS_TESTES.write_text(
    "window.TESTES_DATA = " + json.dumps(df_testes_export.to_dict(orient="records"), ensure_ascii=False) + ";",
    encoding="utf-8"
)
ARQ_JS_RESUMO.write_text(
    "window.RESUMO_DATA = " + json.dumps(df_resumo_export.to_dict(orient="records"), ensure_ascii=False) + ";",
    encoding="utf-8"
)
ARQ_JS_META.write_text(
    "window.META_HTML = " + json.dumps(meta_payload, ensure_ascii=False) + ";",
    encoding="utf-8"
)

rel_js_retornos_parts = [caminho_relativo_html(p, ARQUIVO_HTML_FINAL) for p in ARQ_JS_RETORNOS_PARTS]
rel_js_metricas = caminho_relativo_html(ARQ_JS_METRICAS, ARQUIVO_HTML_FINAL)
rel_js_testes = caminho_relativo_html(ARQ_JS_TESTES, ARQUIVO_HTML_FINAL)
rel_js_resumo = caminho_relativo_html(ARQ_JS_RESUMO, ARQUIVO_HTML_FINAL)
rel_js_meta = caminho_relativo_html(ARQ_JS_META, ARQUIVO_HTML_FINAL)

# Função para juntar js
def montar_scripts_retornos(rel_paths):
    tags = [f'<script src="{html.escape(rel)}"></script>' for rel in rel_paths]

    linhas = ["window.RETORNOS_DATA = []"]
    for i in range(1, len(rel_paths) + 1):
        linhas.append(f"    .concat(window.RETORNOS_DATA_PART{i} || [])")

    script = "<script>\n" + "\n".join(linhas) + ";\n</script>"

    tags.append(script)
    return "\n".join(tags)

# ==========================================================
# 10) GRUPOS DE GRÁFICOS ESTÁTICOS
# ==========================================================
graficos_principais = [
    ("curva_capital.png", "Curva de capital - resumo das estratégias"),
    ("curva_capital_aleatorias_tamanho.png", "Curva de capital - aleatórias por tamanho"),
    ("drawdown.png", "Drawdown - resumo das estratégias"),
    ("drawdown_aleatorias_tamanho.png", "Drawdown - aleatórias por tamanho"),
]

graficos_comparacao = [
    ("dist_retorno.png", "Distribuição de retorno anual"),
    ("dist_sharpe.png", "Distribuição de Sharpe"),
    ("boxplot_retorno.png", "Boxplot de retorno anual"),
    ("boxplot_sharpe.png", "Boxplot de Sharpe"),
    ("pct_bate_ibov.png", "% de carteiras que batem o Ibovespa"),
    ("scatter_risco_retorno.png", "Dispersão risco vs retorno"),
]

bloco_graficos_principais = montar_bloco_graficos(graficos_principais, "Gráficos Principais")
bloco_graficos_comparacao = montar_bloco_graficos(graficos_comparacao, "Distribuições, Comparações e Risco")

# ==========================================================
# 11) CSS
# ==========================================================
css_html = """
body {
    font-family: Arial, Helvetica, sans-serif;
    background: #f7f8fa;
    color: #1f2937;
    margin: 0;
    padding: 0;
}
.container {
    width: 92%;
    max-width: 1600px;
    margin: 0 auto;
    padding: 24px 0 48px 0;
}
.hero {
    background: linear-gradient(135deg, #111827, #1f2937);
    color: white;
    padding: 32px;
    border-radius: 16px;
    margin-bottom: 28px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.15);
}
.hero h1 {
    margin: 0 0 12px 0;
    font-size: 2rem;
}
.hero p {
    margin: 6px 0;
    line-height: 1.6;
}
.nav {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-top: 20px;
}
.nav a {
    text-decoration: none;
    color: white;
    background: rgba(255,255,255,0.12);
    padding: 10px 14px;
    border-radius: 10px;
    font-size: 0.95rem;
}
.section {
    background: white;
    border-radius: 16px;
    padding: 24px;
    margin-bottom: 24px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.08);
}
.section h2 {
    margin-top: 0;
    font-size: 1.5rem;
    color: #111827;
}
.section h3 {
    margin-top: 30px;
    color: #111827;
}
.texto-livre p,
.conclusao p {
    line-height: 1.8;
    color: #374151;
}
.texto-livre ul,
.conclusao ul {
    line-height: 1.8;
    color: #374151;
    padding-left: 22px;
}
.executive-summary {
    display: grid;
    grid-template-columns: 1.35fr 1fr;
    gap: 20px;
    margin: 0 0 24px 0;
}
.exec-main,
.exec-side {
    background: white;
    border-radius: 16px;
    padding: 24px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.08);
}
.exec-main h2,
.exec-side h2 {
    margin-top: 0;
    color: #111827;
}
.highlight-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));
    gap: 16px;
    margin-top: 18px;
}
.highlight-card {
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 16px;
}
.highlight-card .label {
    display: block;
    font-size: 0.85rem;
    color: #6b7280;
    margin-bottom: 8px;
}
.highlight-card .value {
    font-size: 1.05rem;
    font-weight: bold;
    color: #111827;
    line-height: 1.4;
}
.takeaways {
    display: grid;
    grid-template-columns: 1fr;
    gap: 16px;
    margin-top: 18px;
}
.takeaway-card {
    background: #111827;
    color: white;
    border-radius: 14px;
    padding: 18px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.10);
}
.takeaway-card h3 {
    margin-top: 0;
    margin-bottom: 10px;
    font-size: 1rem;
}
.takeaway-card p {
    margin: 0;
    line-height: 1.6;
    color: rgba(255,255,255,0.88);
}
.kpis {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin: 20px 0;
}
.kpi-card {
    background: white;
    border-radius: 16px;
    padding: 20px;
    box-shadow: 0 3px 14px rgba(15, 23, 42, 0.08);
}
.kpi-card h3 {
    margin: 0 0 8px 0;
    font-size: 15px;
    color: #475569;
}
.kpi-card .valor {
    font-size: 24px;
    font-weight: bold;
    color: #0f172a;
}
.filters-grid {
    display: grid;
    grid-template-columns: 1.15fr 1.15fr 0.85fr 0.8fr 0.8fr;
    gap: 14px;
    margin-bottom: 18px;
}
.filter-box label {
    display: block;
    font-size: 13px;
    color: #475569;
    margin-bottom: 6px;
}
.filter-box select {
    width: 100%;
    padding: 10px 12px;
    border-radius: 10px;
    border: 1px solid #cbd5e1;
    background: white;
}
.actions-row {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin: 6px 0 18px 0;
}
.btn-ghost {
    border: 1px solid #cbd5e1;
    background: #fff;
    color: #334155;
    padding: 9px 12px;
    border-radius: 10px;
    cursor: pointer;
    font-size: 13px;
}
.btn-ghost:hover {
    background: #f8fafc;
}
.graficos-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(420px, 1fr));
    gap: 20px;
    margin-bottom: 20px;
}
.grafico-card {
    background: #fafafa;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 16px;
}
.grafico-card h4 {
    margin-top: 0;
    font-size: 1rem;
}
.grafico-img {
    width: 100%;
    border-radius: 10px;
    display: block;
    cursor: zoom-in;
    transition: transform 0.2s ease;
}
.grafico-img:hover {
    transform: scale(1.01);
}
.plotly-wrap {
    min-height: 560px;
}
.table-wrap {
    width: 100%;
    overflow-x: auto;
    overflow-y: hidden;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    background: #fff;
    padding: 8px;
    box-sizing: border-box;
}
.table-wrap.large {
    max-height: 720px;
    overflow-y: auto;
}
.note {
    background: #eff6ff;
    border: 1px solid #bfdbfe;
    border-radius: 12px;
    padding: 14px;
    margin-top: 16px;
}
.small {
    color: #64748b;
    font-size: 13px;
}
.dataTables_wrapper {
    width: 100%;
}
.dataTables_wrapper .dataTables_scroll {
    clear: both;
}
.dataTables_wrapper .dataTables_scrollHeadInner,
.dataTables_wrapper table {
    width: 100% !important;
}
table.dataTable {
    width: 100% !important;
    margin: 0 !important;
}
.footer {
    text-align: center;
    color: #6b7280;
    font-size: 0.9rem;
    margin-top: 32px;
}
#imgModal {
    display: none;
    position: fixed;
    z-index: 9999;
    inset: 0;
    background: rgba(0, 0, 0, 0.88);
    justify-content: center;
    align-items: center;
    padding: 24px;
}
#imgModal.show {
    display: flex;
}
#imgModal img {
    max-width: 95%;
    max-height: 92vh;
    border-radius: 12px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.35);
}
.modal-close {
    position: absolute;
    top: 18px;
    right: 24px;
    color: white;
    font-size: 2rem;
    font-weight: bold;
    cursor: pointer;
    line-height: 1;
}
@media (max-width: 1150px) {
    .kpis, .filters-grid, .executive-summary {
        grid-template-columns: 1fr;
    }
}
"""

# ==========================================================
# 12) JS
# ==========================================================
js_html = """
let RETORNOS_DATA = window.RETORNOS_DATA || [];
let METRICAS_DATA = window.METRICAS_DATA || [];
let TESTES_DATA = window.TESTES_DATA || [];
let RESUMO_DATA = window.RESUMO_DATA || [];
let META_HTML = window.META_HTML || {};

let DT_GRAFICO = null;
let DT_METRICAS = null;
let DT_RESUMO = null;
let DT_TESTES = null;
let DT_METRICAS_FIXA = null;

function uniqueSorted(array, numeric=false) {
    const vals = [...new Set(array.filter(v => v !== null && v !== undefined && v !== ""))];
    if (numeric) {
        return vals.map(x => Number(x)).sort((a,b) => a-b);
    }
    return vals.sort((a,b) => String(a).localeCompare(String(b), 'pt-BR'));
}

function escapeRegex(value) {
    return String(value).replace(/[.*+?^${}()|[\\]\\\\]/g, '\\\\$&');
}

function setSelectedValues(selectId, values) {
    const control = document.getElementById(selectId);
    if (!control || !control.tomselect) return;
    control.tomselect.clear(true);
    values.forEach(v => control.tomselect.addItem(String(v), true));
}

function getSelectedValues(selectId) {
    const control = document.getElementById(selectId);
    if (!control || !control.tomselect) return [];
    return control.tomselect.items || [];
}

function initTomSelect(selectId, options, placeholder) {
    const el = document.getElementById(selectId);
    el.innerHTML = "";

    options.forEach(opt => {
        const option = document.createElement("option");
        option.value = String(opt.value);
        option.textContent = String(opt.label);
        el.appendChild(option);
    });

    return new TomSelect(el, {
        plugins: ['remove_button'],
        maxItems: null,
        maxOptions: 100000,
        persist: false,
        create: false,
        hideSelected: true,
        closeAfterSelect: false,
        placeholder: placeholder,
        searchField: ['text', 'value']
    });
}

function ajustarDataTables() {
    setTimeout(function() {
        $.fn.dataTable.tables({visible: true, api: true}).columns.adjust();
    }, 50);
}

function popularKPIs() {
    const setText = (id, txt) => {
        const el = document.getElementById(id);
        if (el) el.textContent = txt;
    };

    setText("kpi_retorno_tipo", META_HTML.melhor_retorno_tipo || "N/A");
    setText("kpi_retorno_valor", META_HTML.melhor_retorno_valor == null ? "N/A" : (META_HTML.melhor_retorno_valor * 100).toFixed(2) + "%");

    setText("kpi_sharpe_tipo", META_HTML.melhor_sharpe_tipo || "N/A");
    setText("kpi_sharpe_valor", META_HTML.melhor_sharpe_valor == null ? "N/A" : Number(META_HTML.melhor_sharpe_valor).toFixed(3));

    setText("kpi_draw_tipo", META_HTML.melhor_draw_tipo || "N/A");
    setText("kpi_draw_valor", META_HTML.melhor_draw_valor == null ? "N/A" : (META_HTML.melhor_draw_valor * 100).toFixed(2) + "%");

    setText("kpi_perm_valor", META_HTML.perm_p == null ? "N/A" : Number(META_HTML.perm_p).toFixed(4));
}

function tiposDisponiveisSeries() {
    return uniqueSorted(RETORNOS_DATA.map(x => x.tipo), false);
}

function carteirasDisponiveisPorTiposEModo(tiposSelecionados, modosSelecionados) {
    const tipos = tiposSelecionados.length ? tiposSelecionados : tiposDisponiveisSeries();
    const modos = modosSelecionados.length ? modosSelecionados : ["resumo", "individual"];

    const set = new Set();

    RETORNOS_DATA.forEach(row => {
        const okTipo = tipos.includes(row.tipo);
        const okModo = modos.includes(row.modo_visual);

        if (okTipo && okModo) {
            set.add(row.carteira_exibicao);
        }
    });

    return [...set].sort((a,b) => String(a).localeCompare(String(b), 'pt-BR'));
}

function atualizarOpcoesCarteiras() {
    const tipos = getSelectedValues("filtro_tipo");
    const modos = getSelectedValues("filtro_modo_visual");

    const carteirasValidas = carteirasDisponiveisPorTiposEModo(tipos, modos);

    const seletor = document.getElementById("filtro_carteira");
    const selecionadasAtuais = getSelectedValues("filtro_carteira");

    if (seletor.tomselect) {
        seletor.tomselect.clearOptions();

        carteirasValidas.forEach(c => {
            seletor.tomselect.addOption({ value: c, text: c });
        });

        seletor.tomselect.refreshOptions(false);

        const selecionadasFiltradas = selecionadasAtuais.filter(x => carteirasValidas.includes(x));

        seletor.tomselect.clear(true);
        selecionadasFiltradas.forEach(v => seletor.tomselect.addItem(v, true));
    }
}

function filtrarSeries() {
    const tipos = getSelectedValues("filtro_tipo");
    const carteiras = getSelectedValues("filtro_carteira");
    const metrica = document.getElementById("filtro_metrica_serie").value || "retorno_acum";
    const modos = getSelectedValues("filtro_modo_visual");

    let base = RETORNOS_DATA.filter(r => r.metrica === metrica);

    if (tipos.length) {
        base = base.filter(r => tipos.includes(r.tipo));
    }
    if (carteiras.length) {
        base = base.filter(r => carteiras.includes(r.carteira_exibicao));
    }
    if (modos.length) {
        base = base.filter(r => modos.includes(r.modo_visual));
    }

    return base;
}

function atualizarGraficoSeries() {
    const base = filtrarSeries();
    const metrica = document.getElementById("filtro_metrica_serie").value || "retorno_acum";
    const series = uniqueSorted(base.map(x => x.carteira_exibicao), false);

    const traces = series.map(nomeSerie => {
        const serie = base
            .filter(x => x.carteira_exibicao === nomeSerie)
            .sort((a,b) => String(a.data_ref).localeCompare(String(b.data_ref)));

        return {
            x: serie.map(x => x.data_ref),
            y: serie.map(x => Number(x.valor)),
            mode: "lines",
            name: nomeSerie,
            hovertemplate: "Série: %{fullData.name}<br>Data: %{x}<br>Valor: %{y:.6f}<extra></extra>"
        };
    });

    const titulo = metrica === "retorno" ? "Retorno Mensal" : "Retorno Acumulado";

    Plotly.newPlot("plot_series", traces, {
        template: "plotly_white",
        hovermode: "x unified",
        title: "Séries Interativas | " + titulo,
        xaxis: {title: "Data"},
        yaxis: {title: titulo},
        legend: {title: {text: "Séries"}},
        margin: {t: 60, r: 20, l: 60, b: 60}
    }, {responsive: true, displaylogo: false});

    const rowsTabela = base
        .sort((a,b) => {
            if (a.data_ref === b.data_ref) return String(a.carteira_exibicao).localeCompare(String(b.carteira_exibicao), 'pt-BR');
            return String(a.data_ref).localeCompare(String(b.data_ref));
        })
        .map(x => ({
            data_ref: x.data_ref,
            carteira_exibicao: x.carteira_exibicao,
            tipo: x.tipo,
            modo_visual: x.modo_visual,
            metrica: x.metrica,
            valor: Number(x.valor)
        }));

    if (DT_GRAFICO) {
        DT_GRAFICO.clear().rows.add(rowsTabela).draw();
        ajustarDataTables();
    }
}

function filtrarMetricasRanking() {
    const tipos = getSelectedValues("filtro_tipo_ranking");
    const carteiras = getSelectedValues("filtro_carteira_ranking");
    const metrica = document.getElementById("filtro_metrica_ranking").value || "retorno_anual";
    const topn = Number(document.getElementById("filtro_topn").value || 25);

    let base = METRICAS_DATA.slice();

    if (tipos.length) {
        base = base.filter(r => tipos.includes(r.tipo));
    }
    if (carteiras.length) {
        base = base.filter(r => carteiras.includes(r.carteira_exibicao));
    }

    base = base.filter(r => r[metrica] !== null && r[metrica] !== undefined && r[metrica] !== "");

    const ordemAsc = ["volatilidade", "downside_vol", "max_drawdown"].includes(metrica);

    base.sort((a,b) => {
        const va = Number(a[metrica]);
        const vb = Number(b[metrica]);
        return ordemAsc ? va - vb : vb - va;
    });

    return {base: base.slice(0, topn), metrica, ordemAsc};
}

function atualizarGraficoRanking() {
    const {base, metrica} = filtrarMetricasRanking();

    const x = base.map(r => r.carteira_exibicao);
    const y = base.map(r => Number(r[metrica]));
    const custom = base.map(r => r.tipo);

    Plotly.newPlot("plot_ranking", [{
        x: x,
        y: y,
        type: "bar",
        customdata: custom,
        hovertemplate: "Carteira: %{x}<br>Tipo: %{customdata}<br>Valor: %{y:.6f}<extra></extra>"
    }], {
        template: "plotly_white",
        title: "Ranking Interativo por Métrica",
        xaxis: {title: "Carteira", tickangle: -45},
        yaxis: {title: metrica},
        margin: {t: 60, r: 20, l: 60, b: 150}
    }, {responsive: true, displaylogo: false});

    if (DT_METRICAS) {
        DT_METRICAS.clear().rows.add(base).draw();
        ajustarDataTables();
    }
}

function atualizarOpcoesCarteirasRanking() {
    const tipos = getSelectedValues("filtro_tipo_ranking");
    const base = METRICAS_DATA.filter(r => !tipos.length || tipos.includes(r.tipo));
    const carteiras = uniqueSorted(base.map(r => r.carteira_exibicao), false);

    const seletor = document.getElementById("filtro_carteira_ranking");
    const selecionadasAtuais = getSelectedValues("filtro_carteira_ranking");

    if (seletor.tomselect) {
        seletor.tomselect.clearOptions();
        carteiras.forEach(c => seletor.tomselect.addOption({value: c, text: c}));
        seletor.tomselect.refreshOptions(false);

        const selecionadasFiltradas = selecionadasAtuais.filter(x => carteiras.includes(x));
        seletor.tomselect.clear(true);
        selecionadasFiltradas.forEach(v => seletor.tomselect.addItem(v, true));
    }
}

function initTabelaGrafico() {
    DT_GRAFICO = $('#tabela_grafico').DataTable({
        autoWidth: false,
        data: [],
        columns: [
            { data: 'data_ref', title: 'Data' },
            { data: 'carteira_exibicao', title: 'Série' },
            { data: 'tipo', title: 'Tipo' },
            { data: 'modo_visual', title: 'Modo' },
            { data: 'metrica', title: 'Métrica' },
            {
                data: 'valor',
                title: 'Valor',
                render: function(data) {
                    if (data === null || data === undefined || data === "") return "";
                    return Number(data).toFixed(6);
                }
            }
        ],
        deferRender: true,
        processing: true,
        pageLength: 20,
        scrollX: true,
        scrollY: "420px",
        dom: 'Bfrtip',
        buttons: ['copy', 'csv', 'excel'],
        order: [[0, 'asc']],
        initComplete: function() {
            ajustarDataTables();
        }
    });
}

function initTabelaMetricasRanking() {
    DT_METRICAS = $('#tabela_metricas_interativa').DataTable({
        autoWidth: false,
        data: [],
        columns: [
            { data: 'carteira_exibicao', title: 'Carteira' },
            { data: 'tipo', title: 'Tipo' },
            { data: 'retorno_total', title: 'Retorno Total', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'retorno_anual', title: 'Retorno Anual', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'volatilidade', title: 'Volatilidade', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'downside_vol', title: 'Downside Vol', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'sharpe', title: 'Sharpe', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'sortino', title: 'Sortino', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'max_drawdown', title: 'Max Drawdown', render: d => d == null ? "" : Number(d).toFixed(6) },
            { data: 'calmar', title: 'Calmar', render: d => d == null ? "" : Number(d).toFixed(6) }
        ],
        deferRender: true,
        processing: true,
        pageLength: 20,
        scrollX: true,
        scrollY: "460px",
        dom: 'Bfrtip',
        buttons: ['copy', 'csv', 'excel'],
        order: [],
        initComplete: function() {
            ajustarDataTables();
        }
    });
}

function initTabelasFixas() {
    DT_RESUMO = $('#tabela_resumo').DataTable({
        autoWidth: false,
        pageLength: 10,
        scrollX: true,
        scrollY: "320px",
        dom: 'Bfrtip',
        buttons: ['copy', 'csv', 'excel'],
        order: [],
        initComplete: function() {
            ajustarDataTables();
        }
    });

    DT_TESTES = $('#tabela_testes').DataTable({
        autoWidth: false,
        pageLength: 10,
        scrollX: true,
        scrollY: "320px",
        dom: 'Bfrtip',
        buttons: ['copy', 'csv', 'excel'],
        order: [],
        initComplete: function() {
            ajustarDataTables();
        }
    });

    DT_METRICAS_FIXA = $('#tabela_metricas').DataTable({
        autoWidth: false,
        pageLength: 15,
        scrollX: true,
        scrollY: "520px",
        dom: 'Bfrtip',
        buttons: ['copy', 'csv', 'excel'],
        order: [],
        initComplete: function() {
            ajustarDataTables();
        }
    });
}

function selecionarPresetPrincipal() {
    setSelectedValues("filtro_tipo", ["benchmark", "resumo", "setor"]);
    setSelectedValues("filtro_modo_visual", ["resumo", "individual"]);
    
    atualizarOpcoesCarteiras();
    
    setSelectedValues("filtro_carteira", [
        "Ibovespa",
        "ALEATORIA_30_MEDIA",
        "MARKOWITZ_30_MEDIA"
    ]);
    
    document.getElementById("filtro_metrica_serie").value = "retorno_acum";

    setTimeout(function () {
        atualizarGraficoSeries();
    }, 50);
}

function selecionarPresetSecundario() {
    setSelectedValues("filtro_tipo", ["benchmark", "resumo", "setor"]);
    setSelectedValues("filtro_modo_visual", ["resumo", "individual"]);
    
    atualizarOpcoesCarteiras();
    
    setSelectedValues("filtro_carteira", [
        "Ibovespa",
        "ALEATORIA_30_MEDIA",
        "MARKOWITZ_30_MEDIA",
        "Setor | BENS INDUSTRIAIS", 
        "Setor | COMUNICAÇÕES",
        "Setor | CONSUMO CÍCLICO",
        "Setor | CONSUMO NÃO CÍCLICO",
        "Setor | FINANCEIRO",
        "Setor | MATERIAIS BÁSICOS",
        "Setor | PETRÓLEO, GÁS E BIOCOMBUSTÍVEIS", 
        "Setor | SAÚDE",
        "Setor | TECNOLOGIA DA INFORMAÇÃO",
        "Setor | UTILIDADE PÚBLICA"
    ]);
    document.getElementById("filtro_metrica_serie").value = "retorno_acum";

    setTimeout(function () {
        atualizarGraficoSeries();
    }, 50);
}

function selecionarTudoTiposGrafico() {
    setSelectedValues("filtro_tipo", tiposDisponiveisSeries());
    atualizarOpcoesCarteiras();
    atualizarGraficoSeries();
}

function limparCarteirasGrafico() {
    setSelectedValues("filtro_carteira", []);
    atualizarGraficoSeries();
}

document.addEventListener("DOMContentLoaded", function() {
    popularKPIs();

    initTomSelect(
        "filtro_tipo",
        tiposDisponiveisSeries().map(x => ({value: x, label: x})),
        "Selecione os tipos"
    );

    initTomSelect(
        "filtro_carteira",
        uniqueSorted(RETORNOS_DATA.map(x => x.carteira_exibicao), false).map(x => ({value: x, label: x})),
        "Selecione as séries"
    );

    initTomSelect(
        "filtro_tipo_ranking",
        uniqueSorted(METRICAS_DATA.map(x => x.tipo), false).map(x => ({value: x, label: x})),
        "Selecione os tipos"
    );

    initTomSelect(
        "filtro_carteira_ranking",
        uniqueSorted(METRICAS_DATA.map(x => x.carteira_exibicao), false).map(x => ({value: x, label: x})),
        "Selecione as carteiras"
    );

    initTomSelect(
        "filtro_modo_visual",
        [
            {value: "resumo", label: "Resumo"},
            {value: "individual", label: "Individual"}
        ],
        "Selecione o modo"
    );

    initTabelaGrafico();
    initTabelaMetricasRanking();
    initTabelasFixas();

    document.getElementById("filtro_tipo").addEventListener("change", function() {
        atualizarOpcoesCarteiras();
        atualizarGraficoSeries();
    });
    document.getElementById("filtro_carteira").addEventListener("change", atualizarGraficoSeries);
    document.getElementById("filtro_metrica_serie").addEventListener("change", atualizarGraficoSeries);
    document.getElementById("filtro_modo_visual").addEventListener("change", function() {
        atualizarOpcoesCarteiras();
        atualizarGraficoSeries();
    });

    document.getElementById("filtro_tipo_ranking").addEventListener("change", function() {
        atualizarOpcoesCarteirasRanking();
        atualizarGraficoRanking();
    });
    document.getElementById("filtro_carteira_ranking").addEventListener("change", atualizarGraficoRanking);
    document.getElementById("filtro_metrica_ranking").addEventListener("change", atualizarGraficoRanking);
    document.getElementById("filtro_topn").addEventListener("change", atualizarGraficoRanking);

    document.getElementById("btn_preset_principal").addEventListener("click", selecionarPresetPrincipal);
    document.getElementById("btn_preset_secundario").addEventListener("click", selecionarPresetSecundario);

    setSelectedValues("filtro_tipo", ["benchmark", "resumo", "setor"]);
    setSelectedValues("filtro_modo_visual", ["resumo", "individual"]);
    atualizarOpcoesCarteiras();
    setSelectedValues("filtro_carteira", [
        "Ibovespa",
        "ALEATORIA_30_MEDIA",
        "MARKOWITZ_30_MEDIA"
    ]);
    document.getElementById("filtro_metrica_serie").value = "retorno_acum";

    setSelectedValues("filtro_tipo_ranking", ["markowitz", "aleatoria", "benchmark", "setor", "subsetor"]);
    setSelectedValues("filtro_modo_visual", ["resumo", "individual"]);
    atualizarOpcoesCarteirasRanking();
    document.getElementById("filtro_metrica_ranking").value = "sharpe";
    document.getElementById("filtro_topn").value = "25";

    atualizarGraficoSeries();
    atualizarGraficoRanking();
    ajustarDataTables();

    window.addEventListener("resize", ajustarDataTables);

    const modal = document.getElementById("imgModal");
    const modalImg = document.getElementById("modalImg");
    const modalClose = document.getElementById("modalClose");

    document.querySelectorAll(".zoomable-img").forEach(img => {
        img.addEventListener("click", () => {
            modalImg.src = img.src;
            modalImg.alt = img.alt;
            modal.classList.add("show");
        });
    });

    modalClose.addEventListener("click", () => {
        modal.classList.remove("show");
        modalImg.src = "";
    });

    modal.addEventListener("click", (e) => {
        if (e.target === modal) {
            modal.classList.remove("show");
            modalImg.src = "";
        }
    });

    document.addEventListener("keydown", (e) => {
        if (e.key === "Escape") {
            modal.classList.remove("show");
            modalImg.src = "";
        }
    });
});
"""

# ==========================================================
# 13) HTML FINAL
# ==========================================================
html_final = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Análise de Carteiras no Brasil - Relatório Final</title>

    <link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
    <link rel="stylesheet" href="https://cdn.datatables.net/buttons/2.4.2/css/buttons.dataTables.min.css">
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/tom-select@2.3.1/dist/css/tom-select.css">

    <style>
        {css_html}
    </style>
</head>
<body>
    <div class="container">

        <div class="hero">
            <h1>Análise de Carteiras no Brasil - Relatório Final</h1>
            <p>
                Comparação entre carteiras aleatórias, carteiras setoriais, carteiras subsetoriais,
                benchmark do Ibovespa e otimização via Markowitz.
            </p>
            <p>
                O painel combina resultados estáticos e interativos, permitindo alternar métricas,
                selecionar múltiplas séries e cruzar gráfico e tabela ao mesmo tempo.
            </p>

            <div class="nav">
                <a href="#introducao">Introdução</a>
                <a href="#metodologia">Metodologia</a>
                <a href="#series-interativas">Séries Interativas</a>
                <a href="#ranking-interativo">Ranking Interativo</a>
                <a href="#graficos-estaticos">Gráficos Estáticos</a>
                <a href="#estatistica">Estatística</a>
                <a href="#tabelas-finais">Tabelas Finais</a>
                <a href="#conclusao">Conclusão</a>
            </div>
        </div>

        {conteudo_resumo}

        <div class="kpis">
            <div class="kpi-card">
                <h3>Melhor retorno médio</h3>
                <div class="valor" id="kpi_retorno_tipo">-</div>
                <div class="small" id="kpi_retorno_valor">-</div>
            </div>

            <div class="kpi-card">
                <h3>Melhor Sharpe médio</h3>
                <div class="valor" id="kpi_sharpe_tipo">-</div>
                <div class="small" id="kpi_sharpe_valor">-</div>
            </div>

            <div class="kpi-card">
                <h3>Melhor drawdown médio</h3>
                <div class="valor" id="kpi_draw_tipo">-</div>
                <div class="small" id="kpi_draw_valor">-</div>
            </div>

            <div class="kpi-card">
                <h3>Permutação Markowitz vs Aleatória</h3>
                <div class="valor" id="kpi_perm_valor">-</div>
                <div class="small">p-valor</div>
            </div>
        </div>

        <section class="section" id="introducao">
            <h2>Introdução</h2>
            {conteudo_introducao}
        </section>

        <section class="section" id="metodologia">
            <h2>Metodologia</h2>
            {conteudo_metodologia}
        </section>

        <section class="section" id="series-interativas">
            <h2>Séries Interativas</h2>
            <p class="small">
                Por padrão, o gráfico usa séries-resumo para facilitar a leitura. As séries individuais continuam disponíveis no filtro de modo visual.
            </p>

            <div class="filters-grid">
                <div class="filter-box">
                    <label for="filtro_tipo">Tipos de série</label>
                    <select id="filtro_tipo" multiple></select>
                </div>

                <div class="filter-box">
                    <label for="filtro_carteira">Séries</label>
                    <select id="filtro_carteira" multiple></select>
                </div>

                <div class="filter-box">
                    <label for="filtro_modo_visual">Modo visual</label>
                    <select id="filtro_modo_visual" multiple>
                        <option value="resumo" selected>Resumo</option>
                        <option value="individual">Individual</option>
                    </select>
                </div>

                <div class="filter-box">
                    <label for="filtro_metrica_serie">Métrica da série</label>
                    <select id="filtro_metrica_serie">
                        <option value="retorno_acum">Retorno acumulado</option>
                        <option value="retorno">Retorno mensal</option>
                    </select>
                </div>

                <div class="filter-box">
                    <label>Ações rápidas</label>
                    <div class="actions-row">
                        <button class="btn-ghost" id="btn_preset_principal" type="button">Preset principal</button>
                        <button class="btn-ghost" id="btn_preset_secundario" type="button">Preset secundário</button>
                    </div>
                </div>
            </div>

            <div id="plot_series" class="plotly-wrap"></div>

            <h3>Tabela da Base do Gráfico</h3>
            <div class="table-wrap large">
                <table id="tabela_grafico" class="display compact stripe nowrap" style="width:100%"></table>
            </div>
        </section>

        <section class="section" id="ranking-interativo">
            <h2>Ranking Interativo por Métrica</h2>
            <p class="small">
                Selecione a métrica e os grupos desejados. A tabela abaixo mostra os mesmos registros exibidos no ranking.
            </p>

            <div class="filters-grid">
                <div class="filter-box">
                    <label for="filtro_tipo_ranking">Tipos de carteira</label>
                    <select id="filtro_tipo_ranking" multiple></select>
                </div>

                <div class="filter-box">
                    <label for="filtro_carteira_ranking">Carteiras</label>
                    <select id="filtro_carteira_ranking" multiple></select>
                </div>

                <div class="filter-box">
                    <label for="filtro_metrica_ranking">Métrica</label>
                    <select id="filtro_metrica_ranking">
                        <option value="retorno_total">Retorno total</option>
                        <option value="retorno_anual">Retorno anual</option>
                        <option value="volatilidade">Volatilidade</option>
                        <option value="downside_vol">Downside volatility</option>
                        <option value="sharpe">Sharpe</option>
                        <option value="sortino">Sortino</option>
                        <option value="max_drawdown">Max drawdown</option>
                        <option value="calmar">Calmar</option>
                    </select>
                </div>

                <div class="filter-box">
                    <label for="filtro_topn">Top N</label>
                    <select id="filtro_topn">
                        <option value="10">10</option>
                        <option value="25" selected>25</option>
                        <option value="50">50</option>
                        <option value="100">100</option>
                    </select>
                </div>

                <div class="filter-box">
                    <label>Observação</label>
                    <div class="note">
                        Quanto menor o max drawdown e a volatilidade, melhor. Nas demais métricas, quanto maior, melhor.
                    </div>
                </div>
            </div>

            <div id="plot_ranking" class="plotly-wrap"></div>

            <h3>Tabela do Ranking Interativo</h3>
            <div class="table-wrap large">
                <table id="tabela_metricas_interativa" class="display compact stripe nowrap" style="width:100%"></table>
            </div>
        </section>

        <section class="section" id="graficos-estaticos">
            <h2>Gráficos Estáticos</h2>
            {conteudo_resultados}
            {bloco_graficos_principais}
            {bloco_graficos_comparacao}
        </section>

        <section class="section" id="estatistica">
            <h2>Estatística</h2>

            <h3>Resumo das distribuições</h3>
            <div class="table-wrap">
                {tabela_resumo_html}
            </div>

            <h3>Testes estatísticos</h3>
            <div class="table-wrap">
                {tabela_testes_html}
            </div>
        </section>

        <section class="section" id="tabelas-finais">
            <h2>Tabelas Finais</h2>

            <h3>Métricas por carteira</h3>
            <div class="table-wrap large">
                {tabela_metricas_html}
            </div>
        </section>

        <section class="section" id="conclusao">
            {conteudo_conclusao}
        </section>

        <div class="footer">
            <p>HTML gerado automaticamente a partir dos resultados do notebook.</p>
        </div>
    </div>

    <div id="imgModal">
        <span class="modal-close" id="modalClose">&times;</span>
        <img id="modalImg" src="" alt="Imagem ampliada">
    </div>

    <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

    {montar_scripts_retornos(rel_js_retornos_parts)}
    <script src="{rel_js_metricas}"></script>
    <script src="{rel_js_testes}"></script>
    <script src="{rel_js_resumo}"></script>
    <script src="{rel_js_meta}"></script>

    <script src="https://cdn.jsdelivr.net/npm/tom-select@2.3.1/dist/js/tom-select.complete.min.js"></script>

    <script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>
    <script src="https://cdn.datatables.net/buttons/2.4.2/js/dataTables.buttons.min.js"></script>
    <script src="https://cdn.datatables.net/buttons/2.4.2/js/buttons.html5.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/jszip/3.10.1/jszip.min.js"></script>

    <script>
        {js_html}
    </script>
</body>
</html>
"""

# ==========================================================
# 14) SALVAR HTML
# ==========================================================
ARQUIVO_HTML_FINAL.write_text(html_final, encoding="utf-8")

ARQUIVO_HTML_SECUNDARIO = HTML_DIR / "index.html"
ARQUIVO_HTML_SECUNDARIO.write_text(html_final, encoding="utf-8")

print(f"HTML salvo em: {ARQUIVO_HTML_FINAL}")
print(f"Cópia salva em: {ARQUIVO_HTML_SECUNDARIO}")

# ==========================================================
# 15) REGISTRO FINAL
# ==========================================================
df_registro_html = pd.DataFrame([{
    "arquivo_html_raiz": str(ARQUIVO_HTML_FINAL),
    "arquivo_html_resultados": str(ARQUIVO_HTML_SECUNDARIO),
    "js_retornos_part1": str(ARQ_JS_RETORNOS_PARTS[0]),
    "js_retornos_part2": str(ARQ_JS_RETORNOS_PARTS[1]),
    "js_retornos_part3": str(ARQ_JS_RETORNOS_PARTS[2]),
    "js_metricas": str(ARQ_JS_METRICAS),
    "js_testes": str(ARQ_JS_TESTES),
    "js_resumo": str(ARQ_JS_RESUMO),
    "js_meta": str(ARQ_JS_META),
    "status": "gerado_com_sucesso",
    "data_inicio_comum": str(DATA_INICIO_COMUM.date()),
    "data_fim_comum": str(DATA_FIM_COMUM.date())
}])

df_registro_html.to_csv(TABELAS_DIR / "registro_html_final.csv", index=False)

print("\nETAPA 11 FINALIZADA COM SUCESSO.")

ETAPA 11 - HTML FINAL
HTML final: index.html
Assets dir: resultados\html_assets
Período comum do HTML: 2015-01-01 até 2025-12-01
Preparando bases interativas...
Preparando tabelas...
Calculando KPIs e narrativa automática...
Exportando bases JS...
Parte 1 exportada: retornos_data_part1.js | registros: 219,295
Parte 2 exportada: retornos_data_part2.js | registros: 219,295
Parte 3 exportada: retornos_data_part3.js | registros: 219,294
HTML salvo em: index.html
Cópia salva em: resultados\html\index.html

ETAPA 11 FINALIZADA COM SUCESSO.


# Conclusão

Este projeto mostrou que a comparação entre estratégias de construção de carteiras não pode ser reduzida a uma única métrica. Ao analisar simultaneamente retorno, risco e eficiência risco-retorno, fica claro que **retorno puro e qualidade do retorno não apontam necessariamente para o mesmo vencedor**.

As carteiras aleatórias apresentaram retorno anual médio superior ao benchmark, indicando que a diversificação simples, mesmo sem otimização sofisticada, já é capaz de gerar resultados competitivos no mercado. No entanto, parte desse desempenho também está associada ao papel do acaso. Ao considerar um grande conjunto de carteiras possíveis, é natural que algumas combinações apresentem desempenho acima do benchmark. Isso não implica superioridade estrutural da aleatoriedade como estratégia, mas evidencia que a combinação entre diversificação e variabilidade pode gerar bons resultados dentro de um universo amplo de possibilidades.

Por outro lado, as carteiras baseadas em otimização de pesos via modelo de Markowitz não apresentaram o maior retorno médio, mas se destacaram de forma consistente em eficiência risco-retorno. O Sharpe mais elevado e os drawdowns menos severos indicam que essas carteiras foram mais eficientes na alocação de risco, reforçando o papel do modelo como ferramenta de controle e balanceamento, e não necessariamente de maximização de retorno absoluto.

As carteiras setoriais e subsetoriais, por sua vez, apresentaram desempenho inferior em termos de eficiência, com Sharpe mais baixo e drawdowns mais profundos. Esse resultado reforça uma leitura clássica do mercado: a concentração aumenta o risco específico e tende a deteriorar a relação risco-retorno no agregado.

---

📊 Interpretação dos Resultados

Os resultados evidenciam que diferentes estratégias se destacam por razões distintas:

- Carteiras **aleatórias** se destacaram em retorno médio, refletindo a força da diversificação ampla  
- Parte desse desempenho decorre do **efeito do acaso**, que permite que algumas combinações superem o benchmark  
- O modelo de **Markowitz** se destacou em eficiência risco-retorno, com melhor Sharpe e menor drawdown  
- Carteiras **concentradas** (setor e subsetor) apresentaram pior desempenho relativo em termos de risco  
- A comparação entre estratégias depende da métrica analisada: retorno médio e qualidade do retorno podem apontar para conclusões diferentes  

Do ponto de vista econômico, esse resultado é coerente. Estratégias mais agressivas ou amplamente diversificadas tendem a capturar mais retorno médio, enquanto estratégias otimizadas sacrificam parte desse retorno em troca de menor volatilidade e melhor controle de perdas.

---

🧠 Principais Aprendizados

- A **diversificação simples** já se mostrou uma estratégia bastante competitiva em termos de retorno  
- As **carteiras aleatórias** superaram o benchmark no agregado  
- Parte desse desempenho reflete o **efeito do acaso**, e não uma superioridade estrutural da aleatoriedade  
- O modelo de **Markowitz** não maximizou retorno, mas melhorou significativamente a eficiência risco-retorno  
- O Markowitz apresentou **Sharpe mais alto** e **drawdowns menos severos**  
- Carteiras **setoriais e subsetoriais** foram, no agregado, as menos eficientes  
- A concentração aumentou risco sem compensação proporcional em retorno  
- Os testes estatísticos indicam que as diferenças observadas não são apenas ruído aleatório  
- A escolha da estratégia depende do objetivo: **maximizar retorno** ou **otimizar risco-retorno**  

---

🏁 Conclusão Final

O estudo reforça uma ideia central:

**não existe uma única estratégia superior em todas as dimensões analisadas.**

Carteiras aleatórias se destacaram em retorno médio, enquanto o modelo de Markowitz se destacou em eficiência e controle de risco. A diferença entre essas abordagens mostra que a escolha da estratégia deve estar alinhada ao objetivo do investidor.

Além disso, o papel do acaso não deve ser ignorado. Em um universo amplo de carteiras, algumas combinações naturalmente se destacam, o que ajuda a explicar parte do desempenho das carteiras aleatórias. Isso não significa que investir de forma aleatória seja a melhor estratégia, mas sim que a diversificação, combinada com variabilidade, pode produzir resultados competitivos.

Em termos práticos, o projeto sugere que:

**a diversificação é poderosa, a otimização melhora a eficiência, e a melhor estratégia depende do equilíbrio desejado entre retorno e risco.**